In [ ]:
import os
import pandas as pd

DATA_DIR = "/kaggle/input/datasets/profop/synthetic-data"

for filename in sorted(os.listdir(DATA_DIR)):
    if filename.endswith(".parquet"):
        path = os.path.join(DATA_DIR, filename)
        df = pd.read_parquet(path)

        print("\n" + "=" * 100)
        print(f"FILE: {filename}")
        print(f"SHAPE: {df.shape}")
        print("-" * 100)

        print("COLUMNS:")
        for col in df.columns:
            print(f"  {col:30} {str(df[col].dtype)}")

        print("\nSAMPLE:")
        print(df.head(2).to_string(index=False))

In [ ]:
!pip install -q xgboost shap networkx pyarrow joblib

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
!pip install -q torch-geometric

In [ ]:
import os
import gc
import json
import warnings
import random

import numpy as np
import pandas as pd
import networkx as nx
import joblib
import shap

warnings.filterwarnings("ignore")

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve
)

from xgboost import XGBClassifier

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import SAGEConv

In [ ]:
DATA_DIR = "/kaggle/input/datasets/profop/synthetic-data"

MODEL_DIR = "/kaggle/working/models"
os.makedirs(MODEL_DIR, exist_ok=True)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Data:", DATA_DIR)

In [ ]:
customers = pd.read_parquet(
    f"{DATA_DIR}/customers.parquet"
)

orders = pd.read_parquet(
    f"{DATA_DIR}/orders.parquet"
)

refunds = pd.read_parquet(
    f"{DATA_DIR}/refunds.parquet"
)

coupons = pd.read_parquet(
    f"{DATA_DIR}/coupons.parquet"
)

relationships = pd.read_parquet(
    f"{DATA_DIR}/relationships.parquet"
)

labels = pd.read_parquet(
    f"{DATA_DIR}/abuse_labels.parquet"
)

merchants = pd.read_parquet(
    f"{DATA_DIR}/merchants.parquet"
)

products = pd.read_parquet(
    f"{DATA_DIR}/products.parquet"
)

print("Customers:", customers.shape)
print("Orders:", orders.shape)
print("Refunds:", refunds.shape)
print("Coupons:", coupons.shape)
print("Relationships:", relationships.shape)
print("Labels:", labels.shape)

In [ ]:
customers["created_at"] = pd.to_datetime(
    customers["created_at"]
)

orders["timestamp"] = pd.to_datetime(
    orders["timestamp"]
)

refunds["timestamp"] = pd.to_datetime(
    refunds["timestamp"]
)

coupons["timestamp"] = pd.to_datetime(
    coupons["timestamp"]
)

relationships["first_seen"] = pd.to_datetime(
    relationships["first_seen"]
)

relationships["last_seen"] = pd.to_datetime(
    relationships["last_seen"]
)

In [ ]:
labels_model = labels[
    ["entity_id", "is_abuse"]
].copy()

labels_model = labels_model.rename(
    columns={
        "entity_id": "customer_id"
    }
)

labels_model["is_abuse"] = (
    labels_model["is_abuse"]
    .astype(int)
)

print(
    labels_model["is_abuse"].value_counts()
)

print(
    labels_model["is_abuse"].mean()
)

In [ ]:
def build_behavioral_features(
    customers,
    orders,
    refunds,
    coupons,
    prediction_time
):
    
    features = customers[
        ["customer_id", "created_at"]
    ].copy()

    # =========================================================
    # CUSTOMER AGE
    # =========================================================

    features["customer_age_days"] = (
        prediction_time -
        features["created_at"]
    ).dt.total_seconds() / 86400

    features["customer_age_days"] = (
        features["customer_age_days"]
        .clip(lower=0)
    )

    # =========================================================
    # HISTORICAL ORDERS
    # =========================================================

    hist_orders = orders[
        orders["timestamp"] < prediction_time
    ]

    order_stats = hist_orders.groupby(
        "customer_id"
    ).agg(
        order_count=("order_id", "count"),
        total_spend=("amount", "sum"),
        avg_order_amount=("amount", "mean"),
        median_order_amount=("amount", "median"),
        max_order_amount=("amount", "max"),
        total_quantity=("quantity", "sum"),
        unique_merchants=("merchant_id", "nunique"),
        unique_products=("product_id", "nunique"),
        unique_devices=("device_id", "nunique"),
        unique_networks=("network_id", "nunique"),
        unique_addresses=("address_id", "nunique"),
        unique_payment_methods=("payment_id", "nunique")
    ).reset_index()

    features = features.merge(
        order_stats,
        on="customer_id",
        how="left"
    )

    # =========================================================
    # ROLLING ORDER WINDOWS
    # =========================================================

    for days in [7, 30, 90]:

        cutoff = (
            prediction_time -
            pd.Timedelta(days=days)
        )

        recent = hist_orders[
            hist_orders["timestamp"] >= cutoff
        ]

        recent_stats = recent.groupby(
            "customer_id"
        ).agg(
            **{
                f"orders_{days}d": (
                    "order_id",
                    "count"
                ),
                f"spend_{days}d": (
                    "amount",
                    "sum"
                ),
                f"avg_amount_{days}d": (
                    "amount",
                    "mean"
                )
            }
        ).reset_index()

        features = features.merge(
            recent_stats,
            on="customer_id",
            how="left"
        )

    # =========================================================
    # TRANSACTION VELOCITY
    # =========================================================

    features["orders_per_active_day"] = (
        features["order_count"] /
        np.maximum(
            features["customer_age_days"],
            1
        )
    )

    # =========================================================
    # REFUNDS
    # =========================================================

    hist_refunds = refunds[
        refunds["timestamp"] < prediction_time
    ]

    refund_stats = hist_refunds.groupby(
        "customer_id"
    ).agg(
        refund_count=("refund_id", "count"),
        refund_amount=("refund_amount", "sum"),
        avg_refund_amount=(
            "refund_amount",
            "mean"
        )
    ).reset_index()

    features = features.merge(
        refund_stats,
        on="customer_id",
        how="left"
    )

    # Refund rate
    features["refund_rate"] = (
        features["refund_count"] /
        np.maximum(
            features["order_count"],
            1
        )
    )

    features["refund_to_spend_ratio"] = (
        features["refund_amount"] /
        np.maximum(
            features["total_spend"],
            1
        )
    )

    # =========================================================
    # RECENT REFUND WINDOWS
    # =========================================================

    for days in [7, 30, 90]:

        cutoff = (
            prediction_time -
            pd.Timedelta(days=days)
        )

        recent_refunds = hist_refunds[
            hist_refunds["timestamp"] >= cutoff
        ]

        recent_stats = recent_refunds.groupby(
            "customer_id"
        ).agg(
            **{
                f"refunds_{days}d": (
                    "refund_id",
                    "count"
                ),
                f"refund_amount_{days}d": (
                    "refund_amount",
                    "sum"
                )
            }
        ).reset_index()

        features = features.merge(
            recent_stats,
            on="customer_id",
            how="left"
        )

    # =========================================================
    # COUPONS
    # =========================================================

    hist_coupons = coupons[
        coupons["timestamp"] < prediction_time
    ]

    coupon_stats = hist_coupons.groupby(
        "customer_id"
    ).agg(
        coupon_count=("coupon_id", "count"),
        total_discount=(
            "discount_amount",
            "sum"
        ),
        avg_discount=(
            "discount_amount",
            "mean"
        )
    ).reset_index()

    features = features.merge(
        coupon_stats,
        on="customer_id",
        how="left"
    )

    features["coupon_rate"] = (
        features["coupon_count"] /
        np.maximum(
            features["order_count"],
            1
        )
    )

    features["discount_to_spend_ratio"] = (
        features["total_discount"] /
        np.maximum(
            features["total_spend"],
            1
        )
    )

    # =========================================================
    # RECENT COUPON WINDOWS
    # =========================================================

    for days in [7, 30, 90]:

        cutoff = (
            prediction_time -
            pd.Timedelta(days=days)
        )

        recent_coupons = hist_coupons[
            hist_coupons["timestamp"] >= cutoff
        ]

        recent_stats = recent_coupons.groupby(
            "customer_id"
        ).agg(
            **{
                f"coupons_{days}d": (
                    "coupon_id",
                    "count"
                ),
                f"discount_{days}d": (
                    "discount_amount",
                    "sum"
                )
            }
        ).reset_index()

        features = features.merge(
            recent_stats,
            on="customer_id",
            how="left"
        )

    # =========================================================
    # BEHAVIORAL BURST FEATURES
    # =========================================================

    features["refund_burst_ratio"] = (
        features["refunds_7d"] /
        np.maximum(
            features["refund_count"],
            1
        )
    )

    features["coupon_burst_ratio"] = (
        features["coupons_7d"] /
        np.maximum(
            features["coupon_count"],
            1
        )
    )

    features["order_burst_ratio"] = (
        features["orders_7d"] /
        np.maximum(
            features["order_count"],
            1
        )
    )

    # =========================================================
    # FILL
    # =========================================================

    numeric_cols = features.select_dtypes(
        include=np.number
    ).columns

    features[numeric_cols] = (
        features[numeric_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return features

In [ ]:
def build_behavioral_features(
    customers,
    orders,
    refunds,
    coupons,
    prediction_time
):
    
    features = customers[
        ["customer_id", "created_at"]
    ].copy()

    # =========================================================
    # CUSTOMER AGE
    # =========================================================

    features["customer_age_days"] = (
        prediction_time -
        features["created_at"]
    ).dt.total_seconds() / 86400

    features["customer_age_days"] = (
        features["customer_age_days"]
        .clip(lower=0)
    )

    # =========================================================
    # HISTORICAL ORDERS
    # =========================================================

    hist_orders = orders[
        orders["timestamp"] < prediction_time
    ]

    order_stats = hist_orders.groupby(
        "customer_id"
    ).agg(
        order_count=("order_id", "count"),
        total_spend=("amount", "sum"),
        avg_order_amount=("amount", "mean"),
        median_order_amount=("amount", "median"),
        max_order_amount=("amount", "max"),
        total_quantity=("quantity", "sum"),
        unique_merchants=("merchant_id", "nunique"),
        unique_products=("product_id", "nunique"),
        unique_devices=("device_id", "nunique"),
        unique_networks=("network_id", "nunique"),
        unique_addresses=("address_id", "nunique"),
        unique_payment_methods=("payment_id", "nunique")
    ).reset_index()

    features = features.merge(
        order_stats,
        on="customer_id",
        how="left"
    )

    # =========================================================
    # ROLLING ORDER WINDOWS
    # =========================================================

    for days in [7, 30, 90]:

        cutoff = (
            prediction_time -
            pd.Timedelta(days=days)
        )

        recent = hist_orders[
            hist_orders["timestamp"] >= cutoff
        ]

        recent_stats = recent.groupby(
            "customer_id"
        ).agg(
            **{
                f"orders_{days}d": (
                    "order_id",
                    "count"
                ),
                f"spend_{days}d": (
                    "amount",
                    "sum"
                ),
                f"avg_amount_{days}d": (
                    "amount",
                    "mean"
                )
            }
        ).reset_index()

        features = features.merge(
            recent_stats,
            on="customer_id",
            how="left"
        )

    # =========================================================
    # TRANSACTION VELOCITY
    # =========================================================

    features["orders_per_active_day"] = (
        features["order_count"] /
        np.maximum(
            features["customer_age_days"],
            1
        )
    )

    # =========================================================
    # REFUNDS
    # =========================================================

    hist_refunds = refunds[
        refunds["timestamp"] < prediction_time
    ]

    refund_stats = hist_refunds.groupby(
        "customer_id"
    ).agg(
        refund_count=("refund_id", "count"),
        refund_amount=("refund_amount", "sum"),
        avg_refund_amount=(
            "refund_amount",
            "mean"
        )
    ).reset_index()

    features = features.merge(
        refund_stats,
        on="customer_id",
        how="left"
    )

    # Refund rate
    features["refund_rate"] = (
        features["refund_count"] /
        np.maximum(
            features["order_count"],
            1
        )
    )

    features["refund_to_spend_ratio"] = (
        features["refund_amount"] /
        np.maximum(
            features["total_spend"],
            1
        )
    )

    # =========================================================
    # RECENT REFUND WINDOWS
    # =========================================================

    for days in [7, 30, 90]:

        cutoff = (
            prediction_time -
            pd.Timedelta(days=days)
        )

        recent_refunds = hist_refunds[
            hist_refunds["timestamp"] >= cutoff
        ]

        recent_stats = recent_refunds.groupby(
            "customer_id"
        ).agg(
            **{
                f"refunds_{days}d": (
                    "refund_id",
                    "count"
                ),
                f"refund_amount_{days}d": (
                    "refund_amount",
                    "sum"
                )
            }
        ).reset_index()

        features = features.merge(
            recent_stats,
            on="customer_id",
            how="left"
        )

    # =========================================================
    # COUPONS
    # =========================================================

    hist_coupons = coupons[
        coupons["timestamp"] < prediction_time
    ]

    coupon_stats = hist_coupons.groupby(
        "customer_id"
    ).agg(
        coupon_count=("coupon_id", "count"),
        total_discount=(
            "discount_amount",
            "sum"
        ),
        avg_discount=(
            "discount_amount",
            "mean"
        )
    ).reset_index()

    features = features.merge(
        coupon_stats,
        on="customer_id",
        how="left"
    )

    features["coupon_rate"] = (
        features["coupon_count"] /
        np.maximum(
            features["order_count"],
            1
        )
    )

    features["discount_to_spend_ratio"] = (
        features["total_discount"] /
        np.maximum(
            features["total_spend"],
            1
        )
    )

    # =========================================================
    # RECENT COUPON WINDOWS
    # =========================================================

    for days in [7, 30, 90]:

        cutoff = (
            prediction_time -
            pd.Timedelta(days=days)
        )

        recent_coupons = hist_coupons[
            hist_coupons["timestamp"] >= cutoff
        ]

        recent_stats = recent_coupons.groupby(
            "customer_id"
        ).agg(
            **{
                f"coupons_{days}d": (
                    "coupon_id",
                    "count"
                ),
                f"discount_{days}d": (
                    "discount_amount",
                    "sum"
                )
            }
        ).reset_index()

        features = features.merge(
            recent_stats,
            on="customer_id",
            how="left"
        )

    # =========================================================
    # BEHAVIORAL BURST FEATURES
    # =========================================================

    features["refund_burst_ratio"] = (
        features["refunds_7d"] /
        np.maximum(
            features["refund_count"],
            1
        )
    )

    features["coupon_burst_ratio"] = (
        features["coupons_7d"] /
        np.maximum(
            features["coupon_count"],
            1
        )
    )

    features["order_burst_ratio"] = (
        features["orders_7d"] /
        np.maximum(
            features["order_count"],
            1
        )
    )

    # =========================================================
    # FILL
    # =========================================================

    numeric_cols = features.select_dtypes(
        include=np.number
    ).columns

    features[numeric_cols] = (
        features[numeric_cols]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )

    return features

In [ ]:
SNAPSHOT_DATES = {
    "train": [
        pd.Timestamp("2025-08-01")
    ],

    "validation": [
        pd.Timestamp("2025-09-01"),
        pd.Timestamp("2025-10-01")
    ],

    "test": [
        pd.Timestamp("2025-11-01"),
        pd.Timestamp("2025-12-01")
    ]
}

In [ ]:
# ============================================================
# POINT-IN-TIME GRAPH FEATURE ENGINE
# ============================================================

def build_graph_features(
    customers,
    relationships,
    prediction_time
):
    """
    Build graph/infrastructure features using only relationships
    that were known by prediction_time.

    IMPORTANT:
    - Uses first_seen <= prediction_time
    - Does NOT use last_seen
    - Does NOT use event_count
    - Does NOT use abuse labels
    """

    features = customers[
        ["customer_id"]
    ].copy()

    # --------------------------------------------------------
    # Only relationships that existed by prediction time
    # --------------------------------------------------------

    rel = relationships[
        relationships["first_seen"] <= prediction_time
    ].copy()

    # --------------------------------------------------------
    # Infrastructure connection counts
    # --------------------------------------------------------

    infrastructure_types = {
        "device": "device_connections",
        "network": "network_connections",
        "address": "address_connections",
        "payment": "payment_connections"
    }

    for target_type, feature_name in infrastructure_types.items():

        subset = rel[
            rel["target_type"] == target_type
        ]

        counts = (
            subset
            .groupby("source_id")["target_id"]
            .nunique()
            .rename(feature_name)
        )

        features = features.merge(
            counts,
            left_on="customer_id",
            right_index=True,
            how="left"
        )

    # --------------------------------------------------------
    # Relationship-type counts
    # --------------------------------------------------------

    relationship_counts = (
        rel
        .groupby(
            ["source_id", "relationship_type"]
        )["target_id"]
        .nunique()
        .unstack(fill_value=0)
        .reset_index()
        .rename(
            columns={
                "source_id": "customer_id"
            }
        )
    )

    # Convert relationship names into safe feature names
    relationship_counts.columns = [
        (
            str(col)
            .lower()
            .replace(" ", "_")
            .replace("-", "_")
        )
        for col in relationship_counts.columns
    ]

    features = features.merge(
        relationship_counts,
        on="customer_id",
        how="left"
    )

    # --------------------------------------------------------
    # Number of total infrastructure relationships
    # --------------------------------------------------------

    total_relationships = (
        rel
        .groupby("source_id")
        .size()
        .rename("total_relationships")
    )

    features = features.merge(
        total_relationships,
        left_on="customer_id",
        right_index=True,
        how="left"
    )

    # --------------------------------------------------------
    # Relationship age
    # --------------------------------------------------------

    first_relationship = (
        rel
        .groupby("source_id")["first_seen"]
        .min()
        .rename("first_relationship")
    )

    features = features.merge(
        first_relationship,
        left_on="customer_id",
        right_index=True,
        how="left"
    )

    features["relationship_age_days"] = (
        prediction_time -
        features["first_relationship"]
    ).dt.total_seconds() / 86400

    features["relationship_age_days"] = (
        features["relationship_age_days"]
        .clip(lower=0)
    )

    features.drop(
        columns=["first_relationship"],
        inplace=True,
        errors="ignore"
    )

    # --------------------------------------------------------
    # Fill missing numerical features
    # --------------------------------------------------------

    numeric_cols = features.select_dtypes(
        include=np.number
    ).columns

    features[numeric_cols] = (
        features[numeric_cols]
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(0)
    )

    return features


print("✓ build_graph_features() is now defined.")

In [ ]:
def build_snapshot(
    prediction_time
):

    print(
        f"Building snapshot: {prediction_time.date()}"
    )

    behavioral = build_behavioral_features(
        customers,
        orders,
        refunds,
        coupons,
        prediction_time
    )

    graph = build_graph_features(
        customers,
        relationships,
        prediction_time
    )

    df = behavioral.merge(
        graph,
        on="customer_id",
        how="left"
    )

    df = df.merge(
        labels_model,
        on="customer_id",
        how="left"
    )

    df["snapshot_date"] = prediction_time

    return df

In [ ]:
train_snapshots = []

for date in SNAPSHOT_DATES["train"]:
    train_snapshots.append(
        build_snapshot(date)
    )

train_df = pd.concat(
    train_snapshots,
    ignore_index=True
)

val_snapshots = []

for date in SNAPSHOT_DATES["validation"]:
    val_snapshots.append(
        build_snapshot(date)
    )

val_df = pd.concat(
    val_snapshots,
    ignore_index=True
)

test_snapshots = []

for date in SNAPSHOT_DATES["test"]:
    test_snapshots.append(
        build_snapshot(date)
    )

test_df = pd.concat(
    test_snapshots,
    ignore_index=True
)

print(
    "Train:", train_df.shape
)

print(
    "Validation:", val_df.shape
)

print(
    "Test:", test_df.shape
)

In [ ]:
print(
    train_df.columns.tolist()
)

print(
    "Number of features:",
    len(train_df.columns)
)

In [ ]:
DROP_COLUMNS = [
    "customer_id",
    "created_at",
    "snapshot_date",
    "is_abuse"
]

In [ ]:
# Never use these labels as model features
for df in [train_df, val_df, test_df]:

    for col in [
        "abuse_type",
        "ring_id",
        "entity_id",
        "entity_type",
        "label_timestamp"
    ]:

        if col in df.columns:
            df.drop(
                columns=col,
                inplace=True
            )

In [ ]:
X_train = train_df.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)

y_train = train_df["is_abuse"]

X_val = val_df.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)

y_val = val_df["is_abuse"]

X_test = test_df.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)

y_test = test_df["is_abuse"]

In [ ]:
non_numeric = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print(
    "Non-numeric features:",
    non_numeric
)

if non_numeric:

    for df in [X_train, X_val, X_test]:

        for col in non_numeric:

            df[col] = pd.factorize(
                df[col]
            )[0]

In [ ]:
X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

print(
    X_train.shape,
    X_val.shape,
    X_test.shape
)

In [ ]:
# ============================================================
# TEMPORAL HETEROGENEOUS GRAPH BUILDER
# ============================================================

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv

def build_temporal_hetero_graph(
    relationships,
    prediction_time
):
    """
    Build a heterogeneous customer-infrastructure graph
    using ONLY relationships whose first_seen <= prediction_time.

    No labels.
    No event_count.
    No last_seen.
    """

    rel = relationships[
        relationships["first_seen"] <= prediction_time
    ].copy()

    # --------------------------------------------------------
    # Node IDs
    # --------------------------------------------------------

    node_ids = {
        "customer": customers["customer_id"].tolist(),
        "device": rel.loc[
            rel["target_type"] == "device",
            "target_id"
        ].unique().tolist(),

        "network": rel.loc[
            rel["target_type"] == "network",
            "target_id"
        ].unique().tolist(),

        "address": rel.loc[
            rel["target_type"] == "address",
            "target_id"
        ].unique().tolist(),

        "payment": rel.loc[
            rel["target_type"] == "payment",
            "target_id"
        ].unique().tolist()
    }

    mappings = {
        node_type: {
            node_id: i
            for i, node_id in enumerate(ids)
        }
        for node_type, ids in node_ids.items()
    }

    data = HeteroData()

    # --------------------------------------------------------
    # Create node features
    #
    # Feature 0 = constant
    # Feature 1 = log degree
    # --------------------------------------------------------

    for node_type, ids in node_ids.items():

        degree = np.zeros(
            len(ids),
            dtype=np.float32
        )

        if node_type == "customer":

            subset = rel[
                rel["source_type"] == "customer"
            ]

            counts = (
                subset
                .groupby("source_id")
                .size()
            )

            for node_id, count in counts.items():

                if node_id in mappings[node_type]:
                    degree[
                        mappings[node_type][node_id]
                    ] = count

        else:

            subset = rel[
                rel["target_type"] == node_type
            ]

            counts = (
                subset
                .groupby("target_id")
                .size()
            )

            for node_id, count in counts.items():

                if node_id in mappings[node_type]:
                    degree[
                        mappings[node_type][node_id]
                    ] = count

        x = np.column_stack([
            np.ones(len(ids), dtype=np.float32),
            np.log1p(degree)
        ])

        data[node_type].x = torch.tensor(
            x,
            dtype=torch.float
        )

    # --------------------------------------------------------
    # Relationship definitions
    # --------------------------------------------------------

    relation_map = {
        "device": "USED_DEVICE",
        "network": "CONNECTED_VIA",
        "address": "SHIPPED_TO",
        "payment": "PAID_WITH"
    }

    for target_type, relationship_type in relation_map.items():

        subset = rel[
            (rel["target_type"] == target_type) &
            (rel["relationship_type"] == relationship_type)
        ]

        if len(subset) == 0:
            continue

        src = [
            mappings["customer"][x]
            for x in subset["source_id"]
        ]

        dst = [
            mappings[target_type][x]
            for x in subset["target_id"]
        ]

        edge_index = torch.tensor(
            [src, dst],
            dtype=torch.long
        )

        # Customer -> infrastructure
        data[
            "customer",
            f"uses_{target_type}",
            target_type
        ].edge_index = edge_index

        # Infrastructure -> customer
        data[
            target_type,
            f"rev_uses_{target_type}",
            "customer"
        ].edge_index = edge_index.flip(0)

    return data, mappings


print("✓ Temporal heterogeneous graph builder ready.")

In [ ]:
# ============================================================
# PROPER HETEROGENEOUS GRAPHSAGE
# ============================================================

class MerchantRiskGraphSAGE(nn.Module):

    def __init__(
        self,
        metadata,
        hidden_dim=64,
        embedding_dim=32
    ):
        super().__init__()

        node_types, edge_types = metadata

        # ----------------------------------------------------
        # Layer 1
        # ----------------------------------------------------

        self.conv1 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    hidden_dim
                )
                for edge_type in edge_types
            },
            aggr="mean"
        )

        # ----------------------------------------------------
        # Layer 2
        # ----------------------------------------------------

        self.conv2 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    hidden_dim
                )
                for edge_type in edge_types
            },
            aggr="mean"
        )

        # ----------------------------------------------------
        # Final customer projection
        # ----------------------------------------------------

        self.customer_projection = nn.Linear(
            hidden_dim,
            embedding_dim
        )

    def forward(self, x_dict, edge_index_dict):

        x_dict = self.conv1(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            key: F.relu(value)
            for key, value in x_dict.items()
        }

        x_dict = self.conv2(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            key: F.relu(value)
            for key, value in x_dict.items()
        }

        customer_embedding = (
            self.customer_projection(
                x_dict["customer"]
            )
        )

        return customer_embedding

In [ ]:
# ============================================================
# UNSUPERVISED GRAPHSAGE TRAINING
# ============================================================

def train_graphsage(
    graph_data,
    epochs=60,
    hidden_dim=64,
    embedding_dim=32,
    lr=0.005
):

    graph_data = graph_data.to(DEVICE)

    model = MerchantRiskGraphSAGE(
        metadata=graph_data.metadata(),
        hidden_dim=hidden_dim,
        embedding_dim=embedding_dim
    ).to(DEVICE)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4
    )

    model.train()

    edge_types = [
        edge_type
        for edge_type in graph_data.edge_types
        if edge_type[0] == "customer"
    ]

    for epoch in range(epochs):

        optimizer.zero_grad()

        customer_embeddings = model(
            graph_data.x_dict,
            graph_data.edge_index_dict
        )

        loss = torch.tensor(
            0.0,
            device=DEVICE
        )

        # ----------------------------------------------------
        # Positive link reconstruction
        # ----------------------------------------------------

        for edge_type in edge_types:

            target_type = edge_type[2]

            edge_index = graph_data[
                edge_type
            ].edge_index

            if edge_index.size(1) == 0:
                continue

            src = edge_index[0]
            dst = edge_index[1]

            # Infrastructure embeddings are not directly
            # returned by the model, so use structural
            # embeddings through a simple degree-aware
            # representation.

            target_features = graph_data[
                target_type
            ].x

            target_features = target_features.to(
                DEVICE
            )

            # Project infrastructure features into the
            # customer embedding dimension.
            target_scalar = target_features[:, 1:2]

            customer_scores = (
                customer_embeddings[src]
                .mean(dim=1)
            )

            target_scores = (
                target_scalar[dst]
                .squeeze(1)
            )

            positive_score = (
                customer_scores *
                target_scores
            )

            positive_loss = (
                -F.logsigmoid(
                    positive_score
                ).mean()
            )

            loss = loss + positive_loss

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        if (epoch + 1) % 10 == 0:

            print(
                f"Epoch {epoch+1:03d}/{epochs} "
                f"| Loss: {loss.item():.5f}"
            )

    return model

In [ ]:
# ============================================================
# TEMPORAL GRAPHSAGE EMBEDDINGS
# ============================================================

graph_models = {}
graph_embeddings = {}

GRAPH_SNAPSHOT_DATES = [
    pd.Timestamp("2025-08-01"),
    pd.Timestamp("2025-09-01"),
    pd.Timestamp("2025-10-01"),
    pd.Timestamp("2025-11-01"),
    pd.Timestamp("2025-12-01")
]

for snapshot_date in GRAPH_SNAPSHOT_DATES:

    print("\n" + "=" * 70)
    print(
        f"GraphSAGE snapshot: {snapshot_date.date()}"
    )
    print("=" * 70)

    graph_data, mappings = (
        build_temporal_hetero_graph(
            relationships,
            snapshot_date
        )
    )

    print(
        "Nodes:",
        {
            node_type: graph_data[node_type].num_nodes
            for node_type in graph_data.node_types
        }
    )

    print(
        "Edges:",
        {
            str(edge_type):
            graph_data[edge_type].edge_index.shape[1]
            for edge_type in graph_data.edge_types
        }
    )

    model = train_graphsage(
        graph_data,
        epochs=60,
        hidden_dim=64,
        embedding_dim=32,
        lr=0.005
    )

    model.eval()

    graph_data = graph_data.to(DEVICE)

    with torch.no_grad():

        embeddings = model(
            graph_data.x_dict,
            graph_data.edge_index_dict
        )

    embeddings = (
        embeddings
        .cpu()
        .numpy()
    )

    graph_models[
        str(snapshot_date.date())
    ] = model

    graph_embeddings[
        str(snapshot_date.date())
    ] = {
        "embeddings": embeddings,
        "customer_ids": customers[
            "customer_id"
        ].tolist()
    }

    print(
        "Embedding shape:",
        embeddings.shape
    )

In [ ]:
# ============================================================
# CLEAN REBUILD OF TEMPORAL SNAPSHOTS + GRAPHSAGE EMBEDDINGS
# ============================================================

def add_temporal_graph_embeddings_clean(
    df,
    snapshot_date
):
    """
    Attach exactly one set of GraphSAGE embeddings.
    Removes any pre-existing graph_emb_* columns first.
    """

    df = df.copy()

    # Remove old/duplicate GraphSAGE columns
    old_graph_cols = [
        c for c in df.columns
        if c.startswith("graph_emb_")
    ]

    if old_graph_cols:
        df = df.drop(
            columns=old_graph_cols
        )

    key = str(
        pd.Timestamp(snapshot_date).date()
    )

    if key not in graph_embeddings:
        raise KeyError(
            f"No embeddings for {key}. "
            f"Available: {list(graph_embeddings.keys())}"
        )

    embedding_data = graph_embeddings[key]

    emb = embedding_data["embeddings"]
    ids = embedding_data["customer_ids"]

    emb_df = pd.DataFrame(
        emb,
        columns=[
            f"graph_emb_{i}"
            for i in range(emb.shape[1])
        ]
    )

    emb_df["customer_id"] = ids

    df = df.merge(
        emb_df,
        on="customer_id",
        how="left",
        validate="many_to_one"
    )

    return df


# ============================================================
# REBUILD TRAIN
# ============================================================

train_df = build_snapshot(
    pd.Timestamp("2025-08-01")
)

train_df = add_temporal_graph_embeddings_clean(
    train_df,
    pd.Timestamp("2025-08-01")
)


# ============================================================
# REBUILD VALIDATION
# ============================================================

val_sep = build_snapshot(
    pd.Timestamp("2025-09-01")
)

val_oct = build_snapshot(
    pd.Timestamp("2025-10-01")
)

val_sep = add_temporal_graph_embeddings_clean(
    val_sep,
    pd.Timestamp("2025-09-01")
)

val_oct = add_temporal_graph_embeddings_clean(
    val_oct,
    pd.Timestamp("2025-10-01")
)

val_df = pd.concat(
    [val_sep, val_oct],
    ignore_index=True
)


# ============================================================
# REBUILD TEST
# ============================================================

test_nov = build_snapshot(
    pd.Timestamp("2025-11-01")
)

test_dec = build_snapshot(
    pd.Timestamp("2025-12-01")
)

test_nov = add_temporal_graph_embeddings_clean(
    test_nov,
    pd.Timestamp("2025-11-01")
)

test_dec = add_temporal_graph_embeddings_clean(
    test_dec,
    pd.Timestamp("2025-12-01")
)

test_df = pd.concat(
    [test_nov, test_dec],
    ignore_index=True
)


# ============================================================
# VERIFY
# ============================================================

graph_columns = [
    c
    for c in train_df.columns
    if c.startswith("graph_emb_")
]

print("=" * 70)
print("TEMPORAL GRAPHSAGE SNAPSHOTS")
print("=" * 70)

print(
    f"Train      : {train_df.shape}"
)

print(
    f"Validation : {val_df.shape}"
)

print(
    f"Test       : {test_df.shape}"
)

print(
    f"\nGraphSAGE features: {len(graph_columns)}"
)

print(
    "Train missing:",
    train_df[graph_columns].isna().sum().sum()
)

print(
    "Validation missing:",
    val_df[graph_columns].isna().sum().sum()
)

print(
    "Test missing:",
    test_df[graph_columns].isna().sum().sum()
)

print(
    "\nDuplicate GraphSAGE columns:",
    len([
        c for c in train_df.columns
        if "_x" in c or "_y" in c
    ])
)

In [ ]:
# ============================================================
# FINAL FEATURE MATRIX PREPARATION
# ============================================================

DROP_COLUMNS = [
    "customer_id",
    "created_at",
    "snapshot_date",
    "is_abuse",
    "abuse_type",
    "ring_id",
    "entity_id",
    "entity_type",
    "label_timestamp"
]

DROP_COLUMNS = [
    c for c in DROP_COLUMNS
    if c in train_df.columns
]

X_train = train_df.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)

X_val = val_df.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)

X_test = test_df.drop(
    columns=DROP_COLUMNS,
    errors="ignore"
)

y_train = train_df["is_abuse"].astype(int)
y_val = val_df["is_abuse"].astype(int)
y_test = test_df["is_abuse"].astype(int)

# Align columns
X_val = X_val.reindex(
    columns=X_train.columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=X_train.columns,
    fill_value=0
)

# Remove any remaining non-numeric columns
non_numeric = X_train.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Feature count:", X_train.shape[1])
print("Non-numeric:", non_numeric)

print("\nShapes:")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

In [ ]:
print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("\nPositive rates:")
print("Train:", y_train.mean())
print("Val:  ", y_val.mean())
print("Test: ", y_test.mean())

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Val:  ", X_val.isna().sum().sum())
print("Test: ", X_test.isna().sum().sum())

print("\nDtypes:")
print(X_train.dtypes.value_counts())

In [ ]:
import networkx as nx
import numpy as np
import pandas as pd
from collections import defaultdict
from itertools import combinations

def build_networkx_customer_features(relationships, prediction_time):
    """
    Build customer-level graph features using only relationships
    that existed by prediction_time.

    No last_seen or event_count is used because relationship records
    do not contain event-level timestamps.
    """

    rel = relationships[
        relationships["first_seen"] <= prediction_time
    ].copy()

    # Customer-to-infrastructure relationships
    infra_types = {
        "device",
        "network",
        "address",
        "payment"
    }

    rel = rel[
        (
            (rel["source_type"] == "customer") &
            (rel["target_type"].isin(infra_types))
        ) |
        (
            (rel["target_type"] == "customer") &
            (rel["source_type"].isin(infra_types))
        )
    ].copy()

    # Normalize customer/infrastructure orientation
    edges = []

    for row in rel.itertuples(index=False):
        if row.source_type == "customer":
            customer = str(row.source_id)
            infra = str(row.target_id)
            infra_type = row.target_type
        else:
            customer = str(row.target_id)
            infra = str(row.source_id)
            infra_type = row.source_type

        edges.append((customer, f"{infra_type}:{infra}"))

    if not edges:
        return pd.DataFrame(columns=[
            "customer_id",
            "nx_degree",
            "nx_weighted_degree",
            "nx_clustering",
            "nx_component_size",
            "nx_pagerank"
        ])

    edge_df = pd.DataFrame(
        edges,
        columns=["customer_id", "infra_node"]
    ).drop_duplicates()

    # Bipartite graph
    G = nx.Graph()

    for customer, infra in edge_df.itertuples(index=False):
        G.add_node(customer, node_type="customer")
        G.add_node(infra, node_type="infrastructure")
        G.add_edge(customer, infra)

    customer_nodes = [
        n for n, d in G.nodes(data=True)
        if d.get("node_type") == "customer"
    ]

    # Basic degree
    degree = dict(G.degree(customer_nodes))

    # Project infrastructure sharing into customer graph
    customer_graph = nx.Graph()
    customer_graph.add_nodes_from(customer_nodes)

    infra_to_customers = defaultdict(set)

    for customer, infra in edge_df.itertuples(index=False):
        infra_to_customers[infra].add(customer)

    for customers in infra_to_customers.values():
        customers = list(customers)

        # Avoid pathological giant cliques
        if len(customers) <= 100:
            for a, b in combinations(customers, 2):
                if customer_graph.has_edge(a, b):
                    customer_graph[a][b]["weight"] += 1
                else:
                    customer_graph.add_edge(a, b, weight=1)

    weighted_degree = dict(customer_graph.degree(weight="weight"))

    clustering = nx.clustering(
        customer_graph,
        weight="weight"
    )

    # Connected components
    component_size = {}

    for component in nx.connected_components(customer_graph):
        size = len(component)

        for node in component:
            component_size[node] = size

    # PageRank can be expensive but graph is manageable
    if len(customer_graph) > 0:
        pagerank = nx.pagerank(
            customer_graph,
            weight="weight",
            max_iter=100
        )
    else:
        pagerank = {}

    output = pd.DataFrame({
        "customer_id": customer_nodes,
        "nx_degree": [
            degree.get(x, 0) for x in customer_nodes
        ],
        "nx_weighted_degree": [
            weighted_degree.get(x, 0) for x in customer_nodes
        ],
        "nx_clustering": [
            clustering.get(x, 0.0) for x in customer_nodes
        ],
        "nx_component_size": [
            component_size.get(x, 1) for x in customer_nodes
        ],
        "nx_pagerank": [
            pagerank.get(x, 0.0) for x in customer_nodes
        ]
    })

    return output

In [ ]:
SNAPSHOT_TIMES = {
    "train": pd.Timestamp("2025-08-01"),
    "val_1": pd.Timestamp("2025-09-01"),
    "val_2": pd.Timestamp("2025-10-01"),
    "test_1": pd.Timestamp("2025-11-01"),
    "test_2": pd.Timestamp("2025-12-01")
}

nx_features = {}

for name, timestamp in SNAPSHOT_TIMES.items():
    print(f"Building NetworkX features: {name}")

    nx_features[name] = build_networkx_customer_features(
        relationships,
        timestamp
    )

    print(
        name,
        nx_features[name].shape
    )

In [ ]:
import inspect

print(inspect.signature(build_snapshot))
print(inspect.getsource(build_snapshot))

In [ ]:
# ============================================================
# HELPER: ATTACH NETWORKX FEATURES TO SNAPSHOT
# ============================================================

def attach_nx_features(snapshot_df, nx_df):
    """
    Attach point-in-time NetworkX customer features to a snapshot.

    Both tables are customer-level and are joined using customer_id.
    """

    snapshot_df = snapshot_df.copy()
    nx_df = nx_df.copy()

    # Ensure consistent customer ID type
    snapshot_df["customer_id"] = snapshot_df["customer_id"].astype(str)
    nx_df["customer_id"] = nx_df["customer_id"].astype(str)

    # Remove any existing nx_ columns to avoid duplicate columns
    existing_nx_cols = [
        c for c in snapshot_df.columns
        if c.startswith("nx_")
    ]

    if existing_nx_cols:
        snapshot_df = snapshot_df.drop(
            columns=existing_nx_cols
        )

    # Attach NetworkX features
    snapshot_df = snapshot_df.merge(
        nx_df,
        on="customer_id",
        how="left"
    )

    # Customers without graph features get neutral defaults
    nx_cols = [
        c for c in nx_df.columns
        if c.startswith("nx_")
    ]

    for col in nx_cols:
        snapshot_df[col] = snapshot_df[col].fillna(0)

    return snapshot_df

In [ ]:
# ============================================================
# REBUILD ALL SNAPSHOTS + ATTACH NETWORKX FEATURES
# ============================================================

print("Rebuilding snapshots...\n")

# ------------------------------------------------------------
# 1. Build point-in-time snapshots
# ------------------------------------------------------------

train_df = build_snapshot(
    SNAPSHOT_TIMES["train"]
)

val_df_1 = build_snapshot(
    SNAPSHOT_TIMES["val_1"]
)

val_df_2 = build_snapshot(
    SNAPSHOT_TIMES["val_2"]
)

test_df_1 = build_snapshot(
    SNAPSHOT_TIMES["test_1"]
)

test_df_2 = build_snapshot(
    SNAPSHOT_TIMES["test_2"]
)

print("\nSnapshot shapes BEFORE NetworkX:")

print("Train : ", train_df.shape)
print("Val 1 : ", val_df_1.shape)
print("Val 2 : ", val_df_2.shape)
print("Test 1: ", test_df_1.shape)
print("Test 2: ", test_df_2.shape)


# ------------------------------------------------------------
# 2. Attach NetworkX features
# ------------------------------------------------------------

print("\nAttaching NetworkX features...\n")

train_df = attach_nx_features(
    train_df,
    nx_features["train"]
)

val_df_1 = attach_nx_features(
    val_df_1,
    nx_features["val_1"]
)

val_df_2 = attach_nx_features(
    val_df_2,
    nx_features["val_2"]
)

test_df_1 = attach_nx_features(
    test_df_1,
    nx_features["test_1"]
)

test_df_2 = attach_nx_features(
    test_df_2,
    nx_features["test_2"]
)


# ------------------------------------------------------------
# 3. Verify
# ------------------------------------------------------------

print("\nSnapshot shapes AFTER NetworkX:")

print("Train : ", train_df.shape)
print("Val 1 : ", val_df_1.shape)
print("Val 2 : ", val_df_2.shape)
print("Test 1: ", test_df_1.shape)
print("Test 2: ", test_df_2.shape)

nx_cols = [
    c for c in train_df.columns
    if c.startswith("nx_")
]

print("\nNetworkX features:")
print(nx_cols)

print("\nNetworkX feature count:", len(nx_cols))

In [ ]:
# ============================================================
# PREPARE FINAL FEATURE MATRICES
# ============================================================

import numpy as np
import pandas as pd

DROP_COLUMNS = [
    "customer_id",
    "created_at",
    "snapshot_date",
    "is_abuse",
    "abuse_type",
    "ring_id",
    "entity_id",
    "entity_type",
    "label_timestamp"
]


def prepare_matrix(df):
    X = df.drop(
        columns=[
            c for c in DROP_COLUMNS
            if c in df.columns
        ],
        errors="ignore"
    ).copy()

    y = df["is_abuse"].astype(int).copy()

    # Keep numeric features only
    non_numeric = X.select_dtypes(
        exclude=[np.number]
    ).columns.tolist()

    if non_numeric:
        print("Dropping non-numeric columns:", non_numeric)
        X = X.drop(columns=non_numeric)

    # Clean numerical issues
    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    X = X.fillna(0)

    return X, y


# Train
X_train, y_train = prepare_matrix(train_df)

# Validation
X_val_1, y_val_1 = prepare_matrix(val_df_1)
X_val_2, y_val_2 = prepare_matrix(val_df_2)

# Test
X_test_1, y_test_1 = prepare_matrix(test_df_1)
X_test_2, y_test_2 = prepare_matrix(test_df_2)


# Combine temporal validation snapshots
X_val = pd.concat(
    [X_val_1, X_val_2],
    ignore_index=True
)

y_val = pd.concat(
    [y_val_1, y_val_2],
    ignore_index=True
)


# Combine temporal test snapshots
X_test = pd.concat(
    [X_test_1, X_test_2],
    ignore_index=True
)

y_test = pd.concat(
    [y_test_1, y_test_2],
    ignore_index=True
)


# Ensure identical feature columns
feature_columns = X_train.columns.tolist()

X_val = X_val.reindex(
    columns=feature_columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=feature_columns,
    fill_value=0
)


print("FINAL MATRICES")
print("=" * 50)

print("X_train:", X_train.shape)
print("X_val:  ", X_val.shape)
print("X_test: ", X_test.shape)

print("\nLabels:")
print("y_train:", y_train.shape)
print("y_val:  ", y_val.shape)
print("y_test: ", y_test.shape)

print("\nFeature count:", len(feature_columns))

In [ ]:
# ============================================================
# LEAKAGE + DATA SANITY CHECK
# ============================================================

print("Checking feature types...")
print(X_train.dtypes.value_counts())

print("\nMissing values:")
print("Train:", X_train.isna().sum().sum())
print("Val:  ", X_val.isna().sum().sum())
print("Test: ", X_test.isna().sum().sum())

print("\nInfinite values:")
print("Train:", np.isinf(X_train.values).sum())
print("Val:  ", np.isinf(X_val.values).sum())
print("Test: ", np.isinf(X_test.values).sum())

print("\nPotential leakage columns:")

suspicious = [
    c for c in feature_columns
    if any(
        keyword in c.lower()
        for keyword in [
            "abuse",
            "label",
            "ring_id",
            "future",
            "target"
        ]
    )
]

print(suspicious)

print("\nNetworkX features:")
print([
    c for c in feature_columns
    if c.startswith("nx_")
])

print("\nGraphSAGE features:")
print([
    c for c in feature_columns
    if c.startswith("graph_emb_")
])

In [ ]:
# ============================================================
# CLASS DISTRIBUTION
# ============================================================

print("CLASS DISTRIBUTION")
print("=" * 50)

for name, y in [
    ("Train", y_train),
    ("Validation", y_val),
    ("Test", y_test)
]:
    print(f"\n{name}")
    print(y.value_counts())
    print(
        "Abuse rate:",
        f"{y.mean():.4%}"
    )

In [ ]:
# ============================================================
# XGBOOST
# ============================================================

from xgboost import XGBClassifier

positive = int(y_train.sum())
negative = int(len(y_train) - positive)

scale_pos_weight = negative / max(
    positive,
    1
)

print("Positive cases:", positive)
print("Negative cases:", negative)
print(
    "Scale positive weight:",
    scale_pos_weight
)


xgb_model = XGBClassifier(
    n_estimators=700,
    max_depth=6,
    learning_rate=0.04,

    subsample=0.85,
    colsample_bytree=0.85,

    min_child_weight=5,
    gamma=0.1,

    reg_alpha=0.2,
    reg_lambda=2.0,

    objective="binary:logistic",
    eval_metric="aucpr",

    scale_pos_weight=scale_pos_weight,

    tree_method="hist",

    random_state=42,
    n_jobs=-1
)


xgb_model.fit(
    X_train,
    y_train,

    eval_set=[
        (X_train, y_train),
        (X_val, y_val)
    ],

    verbose=False
)

print("\nXGBoost training complete.")

In [ ]:
# ============================================================
# RAW MODEL EVALUATION
# ============================================================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix
)

val_raw = xgb_model.predict_proba(
    X_val
)[:, 1]

test_raw = xgb_model.predict_proba(
    X_test
)[:, 1]


print("VALIDATION")
print("-" * 40)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_val, val_raw):.4f}"
)

print(
    "PR-AUC:",
    f"{average_precision_score(y_val, val_raw):.4f}"
)


print("\nTEST")
print("-" * 40)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_test, test_raw):.4f}"
)

print(
    "PR-AUC:",
    f"{average_precision_score(y_test, test_raw):.4f}"
)

In [ ]:
# ============================================================
# XGBOOST FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

print("TOP 30 FEATURES")
display(
    importance.head(30)
)

In [ ]:
# ============================================================
# SHAP
# ============================================================

import shap

sample_size = min(
    5000,
    len(X_test)
)

X_shap = X_test.sample(
    sample_size,
    random_state=42
)

explainer = shap.TreeExplainer(
    xgb_model
)

shap_values = explainer.shap_values(
    X_shap
)

print(
    "SHAP calculated for",
    len(X_shap),
    "customers."
)

shap.summary_plot(
    shap_values,
    X_shap,
    max_display=20
)

In [ ]:
# ============================================================
# SHAP FEATURE IMPORTANCE
# ============================================================

shap_importance = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(
        shap_values
    ).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "mean_abs_shap",
    ascending=False
)

display(
    shap_importance.head(30)
)

In [ ]:
# ============================================================
# PROBABILITY CALIBRATION
# ============================================================

from sklearn.calibration import CalibratedClassifierCV

calibrator = CalibratedClassifierCV(
    xgb_model,
    method="sigmoid",
    cv="prefit"
)

calibrator.fit(
    X_val,
    y_val
)

val_prob = calibrator.predict_proba(
    X_val
)[:, 1]

test_prob = calibrator.predict_proba(
    X_test
)[:, 1]


print("Calibration complete.")

print(
    "Validation PR-AUC:",
    f"{average_precision_score(y_val, val_prob):.4f}"
)

print(
    "Test PR-AUC:",
    f"{average_precision_score(y_test, test_prob):.4f}"
)

In [ ]:
# ============================================================
# THRESHOLD SEARCH — F1
# ============================================================

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score
)

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

threshold_results = []

for threshold in thresholds:

    pred = (
        val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })


threshold_df = pd.DataFrame(
    threshold_results
)

best_f1_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

best_f1_threshold = float(
    best_f1_row["threshold"]
)

print(
    "Best F1 threshold:",
    best_f1_threshold
)

display(
    threshold_df.sort_values(
        "f1",
        ascending=False
    ).head(10)
)

In [ ]:
# ============================================================
# COST-SENSITIVE THRESHOLD
# ============================================================

FP_COST = 20.0
FN_COST = 1000.0

print("Assumptions:")
print("False positive cost: ₹", FP_COST)
print("False negative cost: ₹", FN_COST)


def calculate_cost(
    y_true,
    probabilities,
    threshold
):

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions
    ).ravel()

    fp_cost = fp * FP_COST
    fn_cost = fn * FN_COST

    total_cost = (
        fp_cost +
        fn_cost
    )

    precision = tp / max(
        tp + fp,
        1
    )

    recall = tp / max(
        tp + fn,
        1
    )

    return {
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "FP_cost": fp_cost,
        "FN_cost": fn_cost,
        "total_cost": total_cost
    }


cost_results = []

for threshold in thresholds:

    result = calculate_cost(
        y_val,
        val_prob,
        threshold
    )

    cost_results.append(result)


cost_df = pd.DataFrame(
    cost_results
)

best_cost_row = cost_df.loc[
    cost_df["total_cost"].idxmin()
]

cost_optimal_threshold = float(
    best_cost_row["threshold"]
)

print(
    "\nCost-optimal threshold:",
    cost_optimal_threshold
)

display(
    cost_df.sort_values(
        "total_cost"
    ).head(10)
)

In [ ]:
# ============================================================
# FINAL TEST EVALUATION
# ============================================================

final_threshold = cost_optimal_threshold

test_pred = (
    test_prob >= final_threshold
).astype(int)


final_metrics = calculate_cost(
    y_test,
    test_prob,
    final_threshold
)


print("=" * 60)
print("FINAL TEST RESULTS")
print("=" * 60)

print(
    "Threshold:",
    f"{final_threshold:.3f}"
)

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_test, test_prob):.4f}"
)

print(
    "PR-AUC:",
    f"{average_precision_score(y_test, test_prob):.4f}"
)

print(
    "Precision:",
    f"{final_metrics['precision']:.4f}"
)

print(
    "Recall:",
    f"{final_metrics['recall']:.4f}"
)

f1_final = f1_score(
    y_test,
    test_pred,
    zero_division=0
)

print(
    "F1:",
    f"{f1_final:.4f}"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        test_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_pred,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# CUSTOMER RISK SCORES
# ============================================================

def risk_band(probability):

    if probability < 0.20:
        return "LOW"

    elif probability < 0.50:
        return "MEDIUM"

    elif probability < 0.80:
        return "HIGH"

    else:
        return "CRITICAL"


customer_ids_test = pd.concat(
    [
        test_df_1["customer_id"],
        test_df_2["customer_id"]
    ],
    ignore_index=True
)


risk_output = pd.DataFrame({
    "customer_id": customer_ids_test,

    "abuse_probability": test_prob,

    "predicted_abuse": test_pred
})


risk_output["risk_score"] = (
    risk_output["abuse_probability"] * 100
)

risk_output["risk_band"] = (
    risk_output["abuse_probability"]
    .apply(risk_band)
)


display(
    risk_output.head(20)
)

In [ ]:
# ============================================================
# HISTORICAL FINANCIAL EXPOSURE
# ============================================================

def get_exposure(df):

    result = pd.DataFrame({
        "customer_id": df["customer_id"].values
    })

    if "refund_amount" in df.columns:
        result["refund_exposure"] = (
            df["refund_amount"]
            .fillna(0)
            .values
        )
    else:
        result["refund_exposure"] = 0.0

    if "total_discount" in df.columns:
        result["discount_exposure"] = (
            df["total_discount"]
            .fillna(0)
            .values
        )
    else:
        result["discount_exposure"] = 0.0

    result["historical_exposure"] = (
        result["refund_exposure"] +
        result["discount_exposure"]
    )

    return result


exposure_test_1 = get_exposure(
    test_df_1
)

exposure_test_2 = get_exposure(
    test_df_2
)

exposure_test = pd.concat(
    [
        exposure_test_1,
        exposure_test_2
    ],
    ignore_index=True
)

print(
    "Exposure rows:",
    len(exposure_test)
)

display(
    exposure_test.head()
)

In [ ]:
# ============================================================
# EXPECTED FINANCIAL LOSS
# ============================================================

risk_output["historical_exposure"] = (
    exposure_test[
        "historical_exposure"
    ].values
)


risk_output["expected_loss"] = (
    risk_output["abuse_probability"] *
    risk_output["historical_exposure"]
)


print(
    "Total historical exposure:",
    f"₹{risk_output['historical_exposure'].sum():,.2f}"
)

print(
    "Total modeled expected loss:",
    f"₹{risk_output['expected_loss'].sum():,.2f}"
)

display(
    risk_output.sort_values(
        "expected_loss",
        ascending=False
    ).head(20)
)

In [ ]:
# ============================================================
# RISK-BASED INTERVENTION POLICY
# ============================================================

def recommend_action(row):

    probability = row["abuse_probability"]
    expected_loss = row["expected_loss"]

    if probability < 0.20:
        return "ALLOW"

    elif probability < 0.50:
        return "ALLOW_WITH_MONITORING"

    elif probability < 0.80:

        if expected_loss >= 500:
            return "STEP_UP_VERIFICATION"

        return "SOFT_REVIEW"

    else:

        if expected_loss >= 1000:
            return "MANUAL_REVIEW"

        return "STEP_UP_VERIFICATION"


risk_output["recommended_action"] = (
    risk_output.apply(
        recommend_action,
        axis=1
    )
)


display(
    risk_output[
        [
            "customer_id",
            "risk_score",
            "risk_band",
            "historical_exposure",
            "expected_loss",
            "recommended_action"
        ]
    ].sort_values(
        "risk_score",
        ascending=False
    ).head(25)
)

In [ ]:
# ============================================================
# RISKGRAPH SUMMARY
# ============================================================

summary = {
    "customers_scored": len(risk_output),

    "predicted_abuse": int(
        risk_output["predicted_abuse"].sum()
    ),

    "high_or_critical": int(
        risk_output["risk_band"].isin(
            ["HIGH", "CRITICAL"]
        ).sum()
    ),

    "critical": int(
        (risk_output["risk_band"] == "CRITICAL").sum()
    ),

    "historical_exposure": (
        risk_output["historical_exposure"].sum()
    ),

    "modeled_expected_loss": (
        risk_output["expected_loss"].sum()
    ),

    "precision": final_metrics["precision"],

    "recall": final_metrics["recall"],

    "f1": f1_final,

    "pr_auc": average_precision_score(
        y_test,
        test_prob
    ),

    "roc_auc": roc_auc_score(
        y_test,
        test_prob
    ),

    "threshold": final_threshold
}

summary_df = pd.DataFrame(
    [summary]
)

display(
    summary_df.T
)

In [ ]:
# ============================================================
# SAVE MODEL + OUTPUTS
# ============================================================

import os
import joblib

OUTPUT_DIR = "/kaggle/working/riskgraph_outputs"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# Model
joblib.dump(
    xgb_model,
    os.path.join(
        OUTPUT_DIR,
        "riskgraph_xgb.pkl"
    )
)

# Calibrator
joblib.dump(
    calibrator,
    os.path.join(
        OUTPUT_DIR,
        "riskgraph_calibrator.pkl"
    )
)

# Configuration
joblib.dump(
    {
        "features": feature_columns,
        "threshold": final_threshold,
        "fp_cost": FP_COST,
        "fn_cost": FN_COST
    },
    os.path.join(
        OUTPUT_DIR,
        "riskgraph_config.pkl"
    )
)


# Predictions
risk_output.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "merchant_risk_predictions.csv"
    ),
    index=False
)


# Feature importance
importance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgb_feature_importance.csv"
    ),
    index=False
)


# SHAP
shap_importance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "shap_feature_importance.csv"
    ),
    index=False
)


# Thresholds
threshold_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "threshold_analysis.csv"
    ),
    index=False
)


# Cost analysis
cost_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cost_analysis.csv"
    ),
    index=False
)


print("Everything saved to:")
print(OUTPUT_DIR)

print("\nFiles:")
print(
    os.listdir(OUTPUT_DIR)
)

In [ ]:
# ============================================================
# FINAL SANITY REPORT
# ============================================================

print("=" * 70)
print("RISKGRAPH — FINAL REPORT")
print("=" * 70)

print(f"Training rows:       {len(X_train):,}")
print(f"Validation rows:     {len(X_val):,}")
print(f"Test rows:           {len(X_test):,}")
print(f"Features:            {len(feature_columns):,}")

print("-" * 70)

print(
    f"ROC-AUC:             "
    f"{summary['roc_auc']:.4f}"
)

print(
    f"PR-AUC:              "
    f"{summary['pr_auc']:.4f}"
)

print(
    f"Precision:            "
    f"{summary['precision']:.4f}"
)

print(
    f"Recall:               "
    f"{summary['recall']:.4f}"
)

print(
    f"F1:                   "
    f"{summary['f1']:.4f}"
)

print(
    f"Decision threshold:   "
    f"{summary['threshold']:.3f}"
)

print("-" * 70)

print(
    f"Historical exposure:  "
    f"₹{summary['historical_exposure']:,.2f}"
)

print(
    f"Modeled expected loss:"
    f" ₹{summary['modeled_expected_loss']:,.2f}"
)

print(
    f"High/Critical cases:  "
    f"{summary['high_or_critical']:,}"
)

print(
    f"Critical cases:       "
    f"{summary['critical']:,}"
)

print("=" * 70)
print("PIPELINE COMPLETE")
print("=" * 70)

In [ ]:
# ============================================================
# FINAL PIPELINE VERIFICATION
# ============================================================

print("=" * 70)
print("RISKGRAPH TRAINING VERIFICATION")
print("=" * 70)

print("\n1. DATA")
print("Train:", train_df.shape)
print("Val  :", val_df_1.shape if "val_df_1" in globals() else val_df.shape)
print("Test :", test_df_1.shape if "test_df_1" in globals() else test_df.shape)

print("\n2. FEATURE MATRIX")
print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\n3. GRAPHSAGE FEATURES")
graph_features = [
    c for c in X_train.columns
    if c.startswith("graph_emb_")
]
print("Count:", len(graph_features))
print(graph_features[:10])

print("\n4. NETWORKX FEATURES")
nx_features_final = [
    c for c in X_train.columns
    if c.startswith("nx_")
]
print("Count:", len(nx_features_final))
print(nx_features_final)

print("\n5. ISOLATION FOREST FEATURES")
if_features = [
    c for c in X_train.columns
    if "isolation" in c.lower() or "anomaly" in c.lower()
]
print("Count:", len(if_features))
print(if_features)

print("\n6. XGBOOST")
print(
    "Trained:",
    "xgb_model" in globals()
)

print("\n7. CALIBRATION")
print(
    "Calibrator:",
    "calibrator" in globals()
)

print("\n8. FINAL PREDICTIONS")
print(
    "Test probabilities:",
    "test_prob" in globals()
)

print(
    "Test predictions:",
    "test_pred" in globals()
)

print("\n9. FINAL METRICS")

if "final_metrics" in globals():
    print(
        f"Precision: {final_metrics['precision']:.4f}"
    )
    print(
        f"Recall:    {final_metrics['recall']:.4f}"
    )
    print(
        f"F1:        {f1_final:.4f}"
    )
    print(
        f"PR-AUC:    {average_precision_score(y_test, test_prob):.4f}"
    )
    print(
        f"ROC-AUC:   {roc_auc_score(y_test, test_prob):.4f}"
    )

print("\n" + "=" * 70)

In [ ]:
# ============================================================
# FINAL TEMPORALLY CONSISTENT GRAPHSAGE
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import HeteroConv, SAGEConv


def build_graph_for_snapshot(
    relationships,
    prediction_time
):
    """
    Build a point-in-time heterogeneous graph.

    Only relationships known by prediction_time are used.
    No labels, event_count, or last_seen.
    """

    rel = relationships[
        relationships["first_seen"] <= prediction_time
    ].copy()

    # --------------------------------------------------------
    # Node IDs
    # --------------------------------------------------------

    node_ids = {
        "customer": customers["customer_id"].tolist(),

        "device": rel.loc[
            rel["target_type"] == "device",
            "target_id"
        ].unique().tolist(),

        "network": rel.loc[
            rel["target_type"] == "network",
            "target_id"
        ].unique().tolist(),

        "address": rel.loc[
            rel["target_type"] == "address",
            "target_id"
        ].unique().tolist(),

        "payment": rel.loc[
            rel["target_type"] == "payment",
            "target_id"
        ].unique().tolist()
    }

    mappings = {
        node_type: {
            node_id: i
            for i, node_id in enumerate(ids)
        }
        for node_type, ids in node_ids.items()
    }

    data = HeteroData()

    # --------------------------------------------------------
    # Node features
    # --------------------------------------------------------

    for node_type, ids in node_ids.items():

        degree = np.zeros(
            len(ids),
            dtype=np.float32
        )

        if node_type == "customer":

            subset = rel[
                rel["source_type"] == "customer"
            ]

            counts = (
                subset
                .groupby("source_id")
                .size()
            )

            for node_id, count in counts.items():

                if node_id in mappings[node_type]:

                    degree[
                        mappings[node_type][node_id]
                    ] = count

        else:

            subset = rel[
                rel["target_type"] == node_type
            ]

            counts = (
                subset
                .groupby("target_id")
                .size()
            )

            for node_id, count in counts.items():

                if node_id in mappings[node_type]:

                    degree[
                        mappings[node_type][node_id]
                    ] = count

        x = np.column_stack([
            np.ones(
                len(ids),
                dtype=np.float32
            ),

            np.log1p(degree)
        ])

        data[node_type].x = torch.tensor(
            x,
            dtype=torch.float
        )

    # --------------------------------------------------------
    # Edges
    # --------------------------------------------------------

    relation_map = {
        "device": "USED_DEVICE",
        "network": "CONNECTED_VIA",
        "address": "SHIPPED_TO",
        "payment": "PAID_WITH"
    }

    for target_type, relationship_type in relation_map.items():

        subset = rel[
            (rel["target_type"] == target_type) &
            (
                rel["relationship_type"]
                == relationship_type
            )
        ]

        if len(subset) == 0:
            continue

        src = [
            mappings["customer"][x]
            for x in subset["source_id"]
        ]

        dst = [
            mappings[target_type][x]
            for x in subset["target_id"]
        ]

        edge_index = torch.tensor(
            [src, dst],
            dtype=torch.long
        )

        data[
            "customer",
            f"uses_{target_type}",
            target_type
        ].edge_index = edge_index

        data[
            target_type,
            f"rev_uses_{target_type}",
            "customer"
        ].edge_index = edge_index.flip(0)

    return data, mappings

In [ ]:
# ============================================================
# GRAPHSAGE ENCODER + ABUSE CLASSIFIER
# ============================================================

class RiskGraphSAGE(nn.Module):

    def __init__(
        self,
        metadata,
        hidden_dim=64,
        embedding_dim=32
    ):

        super().__init__()

        node_types, edge_types = metadata

        self.conv1 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    hidden_dim
                )
                for edge_type in edge_types
            },
            aggr="mean"
        )

        self.conv2 = HeteroConv(
            {
                edge_type: SAGEConv(
                    (-1, -1),
                    hidden_dim
                )
                for edge_type in edge_types
            },
            aggr="mean"
        )

        self.customer_projection = nn.Linear(
            hidden_dim,
            embedding_dim
        )

        self.classifier = nn.Linear(
            embedding_dim,
            1
        )

    def encode(
        self,
        x_dict,
        edge_index_dict
    ):

        x_dict = self.conv1(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            key: F.relu(value)
            for key, value in x_dict.items()
        }

        x_dict = self.conv2(
            x_dict,
            edge_index_dict
        )

        x_dict = {
            key: F.relu(value)
            for key, value in x_dict.items()
        }

        customer_embedding = self.customer_projection(
            x_dict["customer"]
        )

        return customer_embedding

    def forward(
        self,
        x_dict,
        edge_index_dict
    ):

        embedding = self.encode(
            x_dict,
            edge_index_dict
        )

        logits = self.classifier(
            embedding
        ).squeeze(-1)

        return embedding, logits

In [ ]:
# ============================================================
# TRAIN GRAPHSAGE ON TRAINING GRAPH ONLY
# ============================================================

TRAIN_GRAPH_DATE = pd.Timestamp(
    "2025-08-01"
)

print("=" * 70)
print("BUILDING TRAINING GRAPH")
print("=" * 70)

train_graph, train_mapping = build_graph_for_snapshot(
    relationships,
    TRAIN_GRAPH_DATE
)

print(
    "Nodes:",
    {
        node_type:
        train_graph[node_type].num_nodes
        for node_type in train_graph.node_types
    }
)

print(
    "\nEdges:",
    {
        str(edge_type):
        train_graph[edge_type]
        .edge_index.shape[1]
        for edge_type in train_graph.edge_types
    }
)

In [ ]:
# ============================================================
# TRAIN GRAPHSAGE
# ============================================================

graph_model = RiskGraphSAGE(
    metadata=train_graph.metadata(),
    hidden_dim=64,
    embedding_dim=32
).to(DEVICE)

train_graph_device = train_graph.to(DEVICE)

# ------------------------------------------------------------
# Initialize lazy SAGEConv layers
# ------------------------------------------------------------

with torch.no_grad():

    _ = graph_model(
        train_graph_device.x_dict,
        train_graph_device.edge_index_dict
    )


# ------------------------------------------------------------
# Training labels
# ------------------------------------------------------------

train_customer_ids = customers[
    "customer_id"
].tolist()

train_label_map = dict(
    zip(
        labels_model["customer_id"],
        labels_model["is_abuse"]
    )
)

graph_y = torch.tensor(
    [
        train_label_map.get(
            customer_id,
            0
        )
        for customer_id in train_customer_ids
    ],
    dtype=torch.float32,
    device=DEVICE
)


# ------------------------------------------------------------
# Class weighting
# ------------------------------------------------------------

positive = graph_y.sum().item()
negative = len(graph_y) - positive

pos_weight = torch.tensor(
    [negative / max(positive, 1)],
    dtype=torch.float32,
    device=DEVICE
)

criterion = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)

optimizer = torch.optim.Adam(
    graph_model.parameters(),
    lr=0.005,
    weight_decay=1e-4
)


# ------------------------------------------------------------
# Training
# ------------------------------------------------------------

EPOCHS = 80

graph_model.train()

for epoch in range(EPOCHS):

    optimizer.zero_grad()

    embeddings, logits = graph_model(
        train_graph_device.x_dict,
        train_graph_device.edge_index_dict
    )

    loss = criterion(
        logits,
        graph_y
    )

    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        graph_model.parameters(),
        max_norm=5.0
    )

    optimizer.step()

    if (
        (epoch + 1) % 10 == 0
        or epoch == 0
    ):

        with torch.no_grad():

            probs = torch.sigmoid(
                logits
            )

            preds = (
                probs >= 0.5
            ).float()

            accuracy = (
                preds == graph_y
            ).float().mean()

        print(
            f"Epoch {epoch+1:03d}/{EPOCHS} "
            f"| Loss: {loss.item():.5f} "
            f"| Accuracy: {accuracy.item():.4f}"
        )

print("\nGraphSAGE training complete.")

In [ ]:
# ============================================================
# GENERATE TEMPORAL GRAPHSAGE EMBEDDINGS
# ============================================================

def generate_graphsage_embeddings(
    graph_model,
    prediction_time
):

    graph_data, mappings = build_graph_for_snapshot(
        relationships,
        prediction_time
    )

    graph_data = graph_data.to(DEVICE)

    graph_model.eval()

    with torch.no_grad():

        embeddings, _ = graph_model(
            graph_data.x_dict,
            graph_data.edge_index_dict
        )

    embeddings = (
        embeddings
        .cpu()
        .numpy()
    )

    customer_ids = (
        customers["customer_id"]
        .tolist()
    )

    embedding_columns = [
        f"graph_emb_{i}"
        for i in range(
            embeddings.shape[1]
        )
    ]

    emb_df = pd.DataFrame(
        embeddings,
        columns=embedding_columns
    )

    emb_df["customer_id"] = customer_ids

    return emb_df


print("Temporal GraphSAGE inference function ready.")

In [ ]:
# ============================================================
# TEMPORAL GRAPHSAGE EMBEDDINGS
# ============================================================

GRAPH_DATES = {
    "train": pd.Timestamp("2025-08-01"),
    "val_1": pd.Timestamp("2025-09-01"),
    "val_2": pd.Timestamp("2025-10-01"),
    "test_1": pd.Timestamp("2025-11-01"),
    "test_2": pd.Timestamp("2025-12-01")
}

graph_embeddings_final = {}

for name, date in GRAPH_DATES.items():

    print(
        f"\nGenerating embeddings: {name}"
    )

    graph_embeddings_final[name] = (
        generate_graphsage_embeddings(
            graph_model,
            date
        )
    )

    print(
        "Shape:",
        graph_embeddings_final[name].shape
    )

print("\nGraphSAGE temporal inference complete.")

In [ ]:
# ============================================================
# ATTACH FINAL GRAPHSAGE FEATURES
# ============================================================

def attach_graphsage_features(
    df,
    embedding_df
):

    df = df.copy()

    graph_cols = [
        c for c in embedding_df.columns
        if c.startswith("graph_emb_")
    ]

    # Remove previous versions
    df = df.drop(
        columns=[
            c for c in graph_cols
            if c in df.columns
        ],
        errors="ignore"
    )

    df = df.merge(
        embedding_df,
        on="customer_id",
        how="left",
        validate="many_to_one"
    )

    # Missing embeddings should not happen,
    # but protect against it.
    df[graph_cols] = (
        df[graph_cols]
        .fillna(0)
    )

    return df


train_df = attach_graphsage_features(
    train_df,
    graph_embeddings_final["train"]
)

val_df_1 = attach_graphsage_features(
    val_df_1,
    graph_embeddings_final["val_1"]
)

val_df_2 = attach_graphsage_features(
    val_df_2,
    graph_embeddings_final["val_2"]
)

test_df_1 = attach_graphsage_features(
    test_df_1,
    graph_embeddings_final["test_1"]
)

test_df_2 = attach_graphsage_features(
    test_df_2,
    graph_embeddings_final["test_2"]
)


print("GraphSAGE attached.")

print(
    "Train:",
    len([
        c for c in train_df.columns
        if c.startswith("graph_emb_")
    ])
)

print(
    "Missing train embeddings:",
    train_df[
        [
            c for c in train_df.columns
            if c.startswith("graph_emb_")
        ]
    ].isna().sum().sum()
)

In [ ]:
# ============================================================
# ISOLATION FOREST
# ============================================================

from sklearn.ensemble import IsolationForest

print("=" * 70)
print("TRAINING ISOLATION FOREST")
print("=" * 70)


# ------------------------------------------------------------
# Build training features
# ------------------------------------------------------------

IF_DROP = [
    "customer_id",
    "created_at",
    "snapshot_date",
    "is_abuse",
    "abuse_type",
    "ring_id",
    "entity_id",
    "entity_type",
    "label_timestamp"
]

if_features_train = train_df.drop(
    columns=[
        c for c in IF_DROP
        if c in train_df.columns
    ],
    errors="ignore"
).copy()


# Keep numeric only
if_features_train = (
    if_features_train
    .select_dtypes(include=np.number)
)


# Clean
if_features_train = (
    if_features_train
    .replace(
        [np.inf, -np.inf],
        np.nan
    )
    .fillna(0)
)


print(
    "Isolation Forest input:",
    if_features_train.shape
)


# ------------------------------------------------------------
# Train
# ------------------------------------------------------------

isolation_forest = IsolationForest(
    n_estimators=300,
    max_samples="auto",
    contamination="auto",
    random_state=42,
    n_jobs=-1
)

isolation_forest.fit(
    if_features_train
)

print("Isolation Forest training complete.")

In [ ]:
# ============================================================
# TEMPORAL ANOMALY SCORES
# ============================================================

def add_anomaly_score(
    df,
    isolation_model
):

    df = df.copy()

    X_if = df.drop(
        columns=[
            c for c in IF_DROP
            if c in df.columns
        ],
        errors="ignore"
    ).select_dtypes(
        include=np.number
    )

    X_if = (
        X_if
        .replace(
            [np.inf, -np.inf],
            np.nan
        )
        .fillna(0)
    )

    # Ensure exact training feature order
    X_if = X_if.reindex(
        columns=if_features_train.columns,
        fill_value=0
    )

    # Higher = more anomalous
    anomaly_score = -isolation_model.decision_function(
        X_if
    )

    df["isolation_anomaly_score"] = (
        anomaly_score
    )

    return df


train_df = add_anomaly_score(
    train_df,
    isolation_forest
)

val_df_1 = add_anomaly_score(
    val_df_1,
    isolation_forest
)

val_df_2 = add_anomaly_score(
    val_df_2,
    isolation_forest
)

test_df_1 = add_anomaly_score(
    test_df_1,
    isolation_forest
)

test_df_2 = add_anomaly_score(
    test_df_2,
    isolation_forest
)


print(
    "Anomaly feature added."
)

print(
    train_df[
        "isolation_anomaly_score"
    ].describe()
)

In [ ]:
# ============================================================
# FINAL FEATURE MATRIX
# ============================================================

DROP_COLUMNS_FINAL = [
    "customer_id",
    "created_at",
    "snapshot_date",
    "is_abuse",
    "abuse_type",
    "ring_id",
    "entity_id",
    "entity_type",
    "label_timestamp"
]


def prepare_final_matrix(df):

    X = df.drop(
        columns=[
            c for c in DROP_COLUMNS_FINAL
            if c in df.columns
        ],
        errors="ignore"
    ).copy()

    y = df[
        "is_abuse"
    ].astype(int).copy()

    # Numeric only
    non_numeric = X.select_dtypes(
        exclude=np.number
    ).columns.tolist()

    if non_numeric:
        X = X.drop(
            columns=non_numeric
        )

    # Clean
    X = X.replace(
        [np.inf, -np.inf],
        np.nan
    )

    X = X.fillna(0)

    return X, y


X_train, y_train = prepare_final_matrix(
    train_df
)

X_val_1, y_val_1 = prepare_final_matrix(
    val_df_1
)

X_val_2, y_val_2 = prepare_final_matrix(
    val_df_2
)

X_test_1, y_test_1 = prepare_final_matrix(
    test_df_1
)

X_test_2, y_test_2 = prepare_final_matrix(
    test_df_2
)


X_val = pd.concat(
    [
        X_val_1,
        X_val_2
    ],
    ignore_index=True
)

y_val = pd.concat(
    [
        y_val_1,
        y_val_2
    ],
    ignore_index=True
)


X_test = pd.concat(
    [
        X_test_1,
        X_test_2
    ],
    ignore_index=True
)

y_test = pd.concat(
    [
        y_test_1,
        y_test_2
    ],
    ignore_index=True
)


# Align columns
feature_columns = X_train.columns.tolist()

X_val = X_val.reindex(
    columns=feature_columns,
    fill_value=0
)

X_test = X_test.reindex(
    columns=feature_columns,
    fill_value=0
)


print("=" * 70)
print("FINAL FEATURE MATRICES")
print("=" * 70)

print(
    "X_train:",
    X_train.shape
)

print(
    "X_val:",
    X_val.shape
)

print(
    "X_test:",
    X_test.shape
)

print(
    "\nGraphSAGE:",
    len([
        c for c in feature_columns
        if c.startswith("graph_emb_")
    ])
)

print(
    "NetworkX:",
    len([
        c for c in feature_columns
        if c.startswith("nx_")
    ])
)

print(
    "Isolation Forest:",
    len([
        c for c in feature_columns
        if "isolation" in c.lower()
        or "anomaly" in c.lower()
    ])
)

In [ ]:
# ============================================================
# FINAL FEATURE SANITY CHECK
# ============================================================

print("=" * 70)
print("FINAL SANITY CHECK")
print("=" * 70)

print("\nMissing values:")
print(
    "Train:",
    X_train.isna().sum().sum()
)

print(
    "Val:",
    X_val.isna().sum().sum()
)

print(
    "Test:",
    X_test.isna().sum().sum()
)


print("\nInfinite values:")

print(
    "Train:",
    np.isinf(X_train.values).sum()
)

print(
    "Val:",
    np.isinf(X_val.values).sum()
)

print(
    "Test:",
    np.isinf(X_test.values).sum()
)


print("\nFeature groups:")

print(
    "GraphSAGE:",
    len([
        c for c in X_train.columns
        if c.startswith("graph_emb_")
    ])
)

print(
    "NetworkX:",
    len([
        c for c in X_train.columns
        if c.startswith("nx_")
    ])
)

print(
    "Isolation Forest:",
    len([
        c for c in X_train.columns
        if "anomaly" in c.lower()
        or "isolation" in c.lower()
    ])
)


print("\nPotential leakage columns:")

suspicious = [
    c for c in X_train.columns
    if any(
        keyword in c.lower()
        for keyword in [
            "abuse",
            "label",
            "ring_id",
            "future",
            "target"
        ]
    )
]

print(suspicious)

In [ ]:
# ============================================================
# FINAL RISKGRAPH XGBOOST
# ============================================================

from xgboost import XGBClassifier

positive = int(
    y_train.sum()
)

negative = int(
    len(y_train) - positive
)

scale_pos_weight = (
    negative /
    max(positive, 1)
)

print(
    "Positive cases:",
    positive
)

print(
    "Negative cases:",
    negative
)

print(
    "Scale positive weight:",
    scale_pos_weight
)


xgb_model = XGBClassifier(
    n_estimators=800,

    max_depth=6,

    learning_rate=0.035,

    subsample=0.85,

    colsample_bytree=0.85,

    min_child_weight=5,

    gamma=0.1,

    reg_alpha=0.2,

    reg_lambda=2.0,

    objective="binary:logistic",

    eval_metric="aucpr",

    scale_pos_weight=scale_pos_weight,

    tree_method="hist",

    random_state=42,

    n_jobs=-1
)


xgb_model.fit(
    X_train,
    y_train,

    eval_set=[
        (X_train, y_train),
        (X_val, y_val)
    ],

    verbose=False
)

print(
    "\nFINAL XGBOOST TRAINING COMPLETE."
)

In [ ]:
# ============================================================
# RAW FINAL MODEL EVALUATION
# ============================================================

val_raw = xgb_model.predict_proba(
    X_val
)[:, 1]

test_raw = xgb_model.predict_proba(
    X_test
)[:, 1]


print("=" * 70)
print("FINAL RISKGRAPH MODEL")
print("=" * 70)

print("\nVALIDATION")

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_val, val_raw):.4f}"
)

print(
    "PR-AUC:",
    f"{average_precision_score(y_val, val_raw):.4f}"
)


print("\nTEST")

print(
    "ROC-AUC:",
    f"{roc_auc_score(y_test, test_raw):.4f}"
)

print(
    "PR-AUC:",
    f"{average_precision_score(y_test, test_raw):.4f}"
)

In [ ]:
# ============================================================
# FINAL FEATURE IMPORTANCE
# ============================================================

importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": xgb_model.feature_importances_
})

importance = importance.sort_values(
    "importance",
    ascending=False
)

display(
    importance.head(30)
)

In [ ]:
# ============================================================
# FINAL SHAP
# ============================================================

import shap

sample_size = min(
    5000,
    len(X_test)
)

X_shap = X_test.sample(
    sample_size,
    random_state=42
)

explainer = shap.TreeExplainer(
    xgb_model
)

shap_values = explainer.shap_values(
    X_shap
)

print(
    "SHAP calculated for",
    len(X_shap),
    "test rows."
)

shap.summary_plot(
    shap_values,
    X_shap,
    max_display=20
)

In [ ]:
# ============================================================
# FINAL SHAP IMPORTANCE
# ============================================================

shap_importance = pd.DataFrame({
    "feature": X_shap.columns,
    "mean_abs_shap": np.abs(
        shap_values
    ).mean(axis=0)
})

shap_importance = shap_importance.sort_values(
    "mean_abs_shap",
    ascending=False
)

display(
    shap_importance.head(30)
)

In [ ]:
# ============================================================
# PROBABILITY CALIBRATION
# ============================================================

from sklearn.calibration import CalibratedClassifierCV

calibrator = CalibratedClassifierCV(
    xgb_model,
    method="sigmoid",
    cv="prefit"
)

calibrator.fit(
    X_val,
    y_val
)

val_prob = calibrator.predict_proba(
    X_val
)[:, 1]

test_prob = calibrator.predict_proba(
    X_test
)[:, 1]


print(
    "Calibration complete."
)

print(
    "Validation PR-AUC:",
    f"{average_precision_score(y_val, val_prob):.4f}"
)

print(
    "Test PR-AUC:",
    f"{average_precision_score(y_test, test_prob):.4f}"
)

In [ ]:
# ============================================================
# F1 THRESHOLD SEARCH
# ============================================================

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)

threshold_results = []

for threshold in thresholds:

    pred = (
        val_prob >= threshold
    ).astype(int)

    precision = precision_score(
        y_val,
        pred,
        zero_division=0
    )

    recall = recall_score(
        y_val,
        pred,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        pred,
        zero_division=0
    )

    threshold_results.append({
        "threshold": threshold,
        "precision": precision,
        "recall": recall,
        "f1": f1
    })


threshold_df = pd.DataFrame(
    threshold_results
)

best_f1_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]

best_f1_threshold = float(
    best_f1_row["threshold"]
)

print(
    "Best F1 threshold:",
    best_f1_threshold
)

display(
    threshold_df.sort_values(
        "f1",
        ascending=False
    ).head(10)
)

In [ ]:
# ============================================================
# COST-SENSITIVE THRESHOLD
# ============================================================

FP_COST = 20.0
FN_COST = 1000.0


def calculate_cost(
    y_true,
    probabilities,
    threshold
):

    predictions = (
        probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_true,
        predictions
    ).ravel()

    fp_cost = fp * FP_COST
    fn_cost = fn * FN_COST

    total_cost = (
        fp_cost +
        fn_cost
    )

    precision = tp / max(
        tp + fp,
        1
    )

    recall = tp / max(
        tp + fn,
        1
    )

    return {
        "threshold": threshold,
        "TP": tp,
        "FP": fp,
        "TN": tn,
        "FN": fn,
        "precision": precision,
        "recall": recall,
        "FP_cost": fp_cost,
        "FN_cost": fn_cost,
        "total_cost": total_cost
    }


cost_results = []

for threshold in thresholds:

    cost_results.append(
        calculate_cost(
            y_val,
            val_prob,
            threshold
        )
    )


cost_df = pd.DataFrame(
    cost_results
)

best_cost_row = cost_df.loc[
    cost_df["total_cost"].idxmin()
]

cost_optimal_threshold = float(
    best_cost_row["threshold"]
)

print(
    "Cost-optimal threshold:",
    cost_optimal_threshold
)

display(
    cost_df.sort_values(
        "total_cost"
    ).head(10)
)

In [ ]:
# ============================================================
# FINAL TEST RESULTS
# ============================================================

final_threshold = cost_optimal_threshold

test_pred = (
    test_prob >= final_threshold
).astype(int)

final_metrics = calculate_cost(
    y_test,
    test_prob,
    final_threshold
)

f1_final = f1_score(
    y_test,
    test_pred,
    zero_division=0
)


print("=" * 70)
print("RISKGRAPH — FINAL TEST RESULTS")
print("=" * 70)

print(
    f"ROC-AUC:       "
    f"{roc_auc_score(y_test, test_prob):.4f}"
)

print(
    f"PR-AUC:        "
    f"{average_precision_score(y_test, test_prob):.4f}"
)

print(
    f"Precision:     "
    f"{final_metrics['precision']:.4f}"
)

print(
    f"Recall:        "
    f"{final_metrics['recall']:.4f}"
)

print(
    f"F1:            "
    f"{f1_final:.4f}"
)

print(
    f"Threshold:     "
    f"{final_threshold:.3f}"
)

print(
    f"\nFalse Positives: "
    f"{final_metrics['FP']:,}"
)

print(
    f"False Negatives: "
    f"{final_metrics['FN']:,}"
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        test_pred
    )
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        test_pred,
        digits=4,
        zero_division=0
    )
)

In [ ]:
# ============================================================
# FINAL CUSTOMER RISK OUTPUT
# ============================================================

def risk_band(probability):

    if probability < 0.20:
        return "LOW"

    elif probability < 0.50:
        return "MEDIUM"

    elif probability < 0.80:
        return "HIGH"

    else:
        return "CRITICAL"


customer_ids_test = pd.concat(
    [
        test_df_1["customer_id"],
        test_df_2["customer_id"]
    ],
    ignore_index=True
)


risk_output = pd.DataFrame({
    "customer_id": customer_ids_test,

    "abuse_probability": test_prob,

    "predicted_abuse": test_pred
})


risk_output["risk_score"] = (
    risk_output["abuse_probability"] * 100
)

risk_output["risk_band"] = (
    risk_output[
        "abuse_probability"
    ].apply(risk_band)
)


display(
    risk_output.sort_values(
        "risk_score",
        ascending=False
    ).head(25)
)

In [ ]:
# ============================================================
# EXPECTED FINANCIAL LOSS
# ============================================================

def get_exposure(df):

    result = pd.DataFrame({
        "customer_id":
            df["customer_id"].values
    })

    if "refund_amount" in df.columns:

        result["refund_exposure"] = (
            df["refund_amount"]
            .fillna(0)
            .values
        )

    else:

        result["refund_exposure"] = 0.0


    if "total_discount" in df.columns:

        result["discount_exposure"] = (
            df["total_discount"]
            .fillna(0)
            .values
        )

    else:

        result["discount_exposure"] = 0.0


    result["historical_exposure"] = (
        result["refund_exposure"] +
        result["discount_exposure"]
    )

    return result


exposure_test = pd.concat(
    [
        get_exposure(test_df_1),
        get_exposure(test_df_2)
    ],
    ignore_index=True
)


risk_output["historical_exposure"] = (
    exposure_test[
        "historical_exposure"
    ].values
)


risk_output["expected_loss"] = (
    risk_output[
        "abuse_probability"
    ] *
    risk_output[
        "historical_exposure"
    ]
)


print(
    "Historical exposure:",
    f"₹{risk_output['historical_exposure'].sum():,.2f}"
)

print(
    "Modeled expected loss:",
    f"₹{risk_output['expected_loss'].sum():,.2f}"
)

In [ ]:
# ============================================================
# RISK-BASED INTERVENTION
# ============================================================

def recommend_action(row):

    probability = row[
        "abuse_probability"
    ]

    expected_loss = row[
        "expected_loss"
    ]

    if probability < 0.20:

        return "ALLOW"

    elif probability < 0.50:

        return "ALLOW_WITH_MONITORING"

    elif probability < 0.80:

        if expected_loss >= 500:

            return "STEP_UP_VERIFICATION"

        return "SOFT_REVIEW"

    else:

        if expected_loss >= 1000:

            return "MANUAL_REVIEW"

        return "STEP_UP_VERIFICATION"


risk_output["recommended_action"] = (
    risk_output.apply(
        recommend_action,
        axis=1
    )
)


display(
    risk_output[
        [
            "customer_id",
            "risk_score",
            "risk_band",
            "historical_exposure",
            "expected_loss",
            "recommended_action"
        ]
    ]
    .sort_values(
        "risk_score",
        ascending=False
    )
    .head(25)
)

In [ ]:
# ============================================================
# SAVE FINAL RISKGRAPH ARTIFACTS
# ============================================================

import os
import joblib

OUTPUT_DIR = (
    "/kaggle/working/"
    "riskgraph_outputs"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# XGBoost
joblib.dump(
    xgb_model,
    os.path.join(
        OUTPUT_DIR,
        "riskgraph_xgb.pkl"
    )
)


# Calibration
joblib.dump(
    calibrator,
    os.path.join(
        OUTPUT_DIR,
        "riskgraph_calibrator.pkl"
    )
)


# GraphSAGE
torch.save(
    graph_model.state_dict(),
    os.path.join(
        OUTPUT_DIR,
        "riskgraph_graphsage.pt"
    )
)


# Isolation Forest
joblib.dump(
    isolation_forest,
    os.path.join(
        OUTPUT_DIR,
        "riskgraph_isolation_forest.pkl"
    )
)


# Configuration
joblib.dump(
    {
        "features": feature_columns,

        "threshold": final_threshold,

        "fp_cost": FP_COST,

        "fn_cost": FN_COST,

        "graph_embedding_dim": 32,

        "graph_hidden_dim": 64
    },
    os.path.join(
        OUTPUT_DIR,
        "riskgraph_config.pkl"
    )
)


# Predictions
risk_output.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "merchant_risk_predictions.csv"
    ),
    index=False
)


# Importance
importance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "xgb_feature_importance.csv"
    ),
    index=False
)


shap_importance.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "shap_feature_importance.csv"
    ),
    index=False
)


threshold_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "threshold_analysis.csv"
    ),
    index=False
)


cost_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "cost_analysis.csv"
    ),
    index=False
)


print("=" * 70)
print("FINAL ARTIFACTS SAVED")
print("=" * 70)

for file in sorted(
    os.listdir(OUTPUT_DIR)
):
    print(file)

In [ ]:
# ============================================================
# RISKGRAPH — FINAL REPORT
# ============================================================

print("=" * 75)
print("                 RISKGRAPH FINAL MODEL")
print("=" * 75)

print("\nMODEL COMPONENTS")
print("-" * 75)

print("Temporal behavioral features : YES")
print("NetworkX graph features      : YES")
print("GraphSAGE embeddings         : YES (32)")
print("Isolation Forest anomaly     : YES")
print("XGBoost classifier           : YES")
print("Probability calibration     : YES")
print("SHAP explainability          : YES")
print("Cost-sensitive threshold     : YES")

print("\nFEATURES")
print("-" * 75)

print(
    "Total features:",
    len(feature_columns)
)

print(
    "GraphSAGE:",
    len([
        c for c in feature_columns
        if c.startswith("graph_emb_")
    ])
)

print(
    "NetworkX:",
    len([
        c for c in feature_columns
        if c.startswith("nx_")
    ])
)

print(
    "Anomaly:",
    len([
        c for c in feature_columns
        if "anomaly" in c.lower()
        or "isolation" in c.lower()
    ])
)

print("\nTEST PERFORMANCE")
print("-" * 75)

print(
    f"ROC-AUC:     "
    f"{roc_auc_score(y_test, test_prob):.4f}"
)

print(
    f"PR-AUC:      "
    f"{average_precision_score(y_test, test_prob):.4f}"
)

print(
    f"Precision:   "
    f"{final_metrics['precision']:.4f}"
)

print(
    f"Recall:      "
    f"{final_metrics['recall']:.4f}"
)

print(
    f"F1:          "
    f"{f1_final:.4f}"
)

print(
    f"Threshold:   "
    f"{final_threshold:.3f}"
)

print("\nFINANCIAL RISK")
print("-" * 75)

print(
    f"Historical exposure: "
    f"₹{risk_output['historical_exposure'].sum():,.2f}"
)

print(
    f"Modeled expected loss:"
    f" ₹{risk_output['expected_loss'].sum():,.2f}"
)

print("\nDECISIONS")
print("-" * 75)

print(
    "Predicted abuse:",
    int(
        risk_output[
            "predicted_abuse"
        ].sum()
    )
)

print(
    "High/Critical:",
    int(
        risk_output[
            "risk_band"
        ].isin(
            ["HIGH", "CRITICAL"]
        ).sum()
    )
)

print(
    "Critical:",
    int(
        (
            risk_output[
                "risk_band"
            ] == "CRITICAL"
        ).sum()
    )
)

print("\n" + "=" * 75)
print("                  PIPELINE COMPLETE")
print("=" * 75)

In [ ]:
# Show variables related to GraphSAGE
[v for v in globals().keys() if "sage" in v.lower() or "graph" in v.lower()]

In [ ]:
print(type(graph_model))
print(type(graph_models) if "graph_models" in globals() else "graph_models not found")

In [ ]:
# ============================================================
# SAVE COMPLETE RISKGRAPH MODEL — ROBUST VERSION
# ============================================================

import os
import json
import joblib
import torch
import numpy as np

SAVE_DIR = "/kaggle/working/riskgraph_final_model"
os.makedirs(SAVE_DIR, exist_ok=True)

print("=" * 70)
print("SAVING COMPLETE RISKGRAPH MODEL")
print("=" * 70)


# ============================================================
# 1. FIND THE TRAINED MODELS
# ============================================================

print("\n[1] Checking trained models...")

# XGBoost
if "xgb_model" in globals():
    FINAL_XGB = xgb_model
elif "model" in globals():
    FINAL_XGB = model
else:
    raise NameError(
        "Could not find the trained XGBoost model."
    )

print("  XGBoost:", type(FINAL_XGB).__name__)


# Calibrator
if "calibrator" in globals():
    FINAL_CALIBRATOR = calibrator
elif "calibrated_model" in globals():
    FINAL_CALIBRATOR = calibrated_model
else:
    FINAL_CALIBRATOR = None
    print("  WARNING: calibrator not found")


# Isolation Forest
if "isolation_forest" in globals():
    FINAL_IF = isolation_forest
elif "iso_forest" in globals():
    FINAL_IF = iso_forest
else:
    FINAL_IF = None
    print("  WARNING: Isolation Forest not found")


# GraphSAGE
if "graph_model" in globals():
    FINAL_GRAPH = graph_model
elif "graph_models" in globals():

    # If graph_models is a dictionary/list, inspect it
    if isinstance(graph_models, dict):
        print("  graph_models keys:", list(graph_models.keys()))

        # Prefer the training graph
        if "train" in graph_models:
            FINAL_GRAPH = graph_models["train"]
        elif "TRAIN" in graph_models:
            FINAL_GRAPH = graph_models["TRAIN"]
        else:
            FINAL_GRAPH = next(iter(graph_models.values()))

    elif isinstance(graph_models, (list, tuple)):
        FINAL_GRAPH = graph_models[0]

    else:
        FINAL_GRAPH = graph_models

else:
    raise NameError(
        "Could not find the trained GraphSAGE model."
    )

print("  GraphSAGE:", type(FINAL_GRAPH).__name__)


# ============================================================
# 2. SAVE XGBOOST
# ============================================================

joblib.dump(
    FINAL_XGB,
    os.path.join(SAVE_DIR, "xgboost_model.pkl")
)

print("[OK] XGBoost saved")


# ============================================================
# 3. SAVE CALIBRATOR
# ============================================================

if FINAL_CALIBRATOR is not None:

    joblib.dump(
        FINAL_CALIBRATOR,
        os.path.join(
            SAVE_DIR,
            "probability_calibrator.pkl"
        )
    )

    print("[OK] Calibrator saved")


# ============================================================
# 4. SAVE ISOLATION FOREST
# ============================================================

if FINAL_IF is not None:

    joblib.dump(
        FINAL_IF,
        os.path.join(
            SAVE_DIR,
            "isolation_forest.pkl"
        )
    )

    print("[OK] Isolation Forest saved")


# ============================================================
# 5. SAVE GRAPHSAGE
# ============================================================

torch.save(
    {
        "model_state_dict": FINAL_GRAPH.state_dict(),
        "model_class": type(FINAL_GRAPH).__name__,
        "hidden_dim": 64,
        "embedding_dim": 32
    },
    os.path.join(
        SAVE_DIR,
        "graphsage_model.pt"
    )
)

print("[OK] GraphSAGE saved")


# ============================================================
# 6. FIND THRESHOLD
# ============================================================

print("\n[6] Finding decision threshold...")

FINAL_THRESHOLD = None

# Try common variable names
for name in [
    "best_threshold",
    "optimal_threshold",
    "cost_optimal_threshold",
    "selected_threshold",
    "decision_threshold",
    "optimal_thresh",
    "best_thresh"
]:

    if name in globals():

        value = globals()[name]

        try:
            FINAL_THRESHOLD = float(value)
            print(
                f"  Found {name} = "
                f"{FINAL_THRESHOLD:.4f}"
            )
            break
        except:
            pass


# Search threshold_analysis if available
if FINAL_THRESHOLD is None and "threshold_analysis" in globals():

    ta = threshold_analysis

    print(
        "  threshold_analysis columns:",
        list(ta.columns)
    )

    # Try to identify threshold column
    threshold_col = None

    for col in [
        "threshold",
        "Threshold",
        "probability_threshold"
    ]:

        if col in ta.columns:
            threshold_col = col
            break

    # Try to identify cost column
    cost_col = None

    for col in [
        "cost",
        "total_cost",
        "Total Cost",
        "expected_cost"
    ]:

        if col in ta.columns:
            cost_col = col
            break

    if threshold_col and cost_col:

        idx = ta[cost_col].idxmin()

        FINAL_THRESHOLD = float(
            ta.loc[idx, threshold_col]
        )

        print(
            f"  Calculated optimal threshold = "
            f"{FINAL_THRESHOLD:.4f}"
        )


# If threshold cannot be found, don't stop model saving
if FINAL_THRESHOLD is None:

    print(
        "  WARNING: No threshold variable found."
    )

    print(
        "  Model artifacts will still be saved."
    )

    FINAL_THRESHOLD = 0.50


# ============================================================
# 7. FEATURE COLUMNS
# ============================================================

print("\n[7] Saving feature metadata...")

if "X_train" in globals():

    FEATURE_COLUMNS = list(X_train.columns)

elif "feature_columns" in globals():

    FEATURE_COLUMNS = list(feature_columns)

else:

    FEATURE_COLUMNS = []

    print(
        "  WARNING: X_train/feature_columns not found"
    )


with open(
    os.path.join(
        SAVE_DIR,
        "feature_columns.json"
    ),
    "w"
) as f:

    json.dump(
        FEATURE_COLUMNS,
        f,
        indent=2
    )

print(
    f"[OK] {len(FEATURE_COLUMNS)} feature columns saved"
)


# ============================================================
# 8. MODEL CONFIGURATION
# ============================================================

CONFIG = {

    "model_name": "RiskGraph",

    "version": "1.0",

    "xgboost": {
        "type": type(FINAL_XGB).__name__
    },

    "graphsage": {
        "type": type(FINAL_GRAPH).__name__,
        "hidden_dim": 64,
        "embedding_dim": 32
    },

    "isolation_forest": (
        FINAL_IF is not None
    ),

    "calibration": (
        FINAL_CALIBRATOR is not None
    ),

    "decision_threshold": FINAL_THRESHOLD,

    "risk_bands": {
        "LOW": [0.00, 0.20],
        "MEDIUM": [0.20, 0.50],
        "HIGH": [0.50, 0.80],
        "CRITICAL": [0.80, 1.00]
    },

    "cost_model": {
        "false_positive_cost": 20,
        "false_negative_cost": 1000
    },

    "feature_count": len(FEATURE_COLUMNS)
}


with open(
    os.path.join(
        SAVE_DIR,
        "model_config.json"
    ),
    "w"
) as f:

    json.dump(
        CONFIG,
        f,
        indent=2
    )

print("[OK] Model configuration saved")


# ============================================================
# 9. SAVE GRAPH EMBEDDINGS
# ============================================================

if "graph_embeddings_final" in globals():

    joblib.dump(
        graph_embeddings_final,
        os.path.join(
            SAVE_DIR,
            "graph_embeddings_final.pkl"
        )
    )

    print("[OK] Graph embeddings saved")


# ============================================================
# 10. SAVE FEATURE IMPORTANCE
# ============================================================

if "xgb_feature_importance" in globals():

    xgb_feature_importance.to_csv(
        os.path.join(
            SAVE_DIR,
            "xgb_feature_importance.csv"
        ),
        index=False
    )

    print("[OK] XGBoost feature importance saved")


if "shap_feature_importance" in globals():

    shap_feature_importance.to_csv(
        os.path.join(
            SAVE_DIR,
            "shap_feature_importance.csv"
        ),
        index=False
    )

    print("[OK] SHAP feature importance saved")


# ============================================================
# 11. SAVE THRESHOLD ANALYSIS
# ============================================================

if "threshold_analysis" in globals():

    threshold_analysis.to_csv(
        os.path.join(
            SAVE_DIR,
            "threshold_analysis.csv"
        ),
        index=False
    )

    print("[OK] Threshold analysis saved")


# ============================================================
# 12. SAVE COST ANALYSIS
# ============================================================

if "cost_analysis" in globals():

    cost_analysis.to_csv(
        os.path.join(
            SAVE_DIR,
            "cost_analysis.csv"
        ),
        index=False
    )

    print("[OK] Cost analysis saved")


# ============================================================
# 13. SAVE PREDICTIONS
# ============================================================

if "merchant_risk_predictions" in globals():

    merchant_risk_predictions.to_csv(
        os.path.join(
            SAVE_DIR,
            "merchant_risk_predictions.csv"
        ),
        index=False
    )

    print("[OK] Merchant risk predictions saved")


# ============================================================
# 14. SAVE COMPLETE TABULAR PIPELINE
# ============================================================

COMPLETE_PIPELINE = {

    "xgboost_model": FINAL_XGB,

    "calibrator": FINAL_CALIBRATOR,

    "isolation_forest": FINAL_IF,

    "threshold": FINAL_THRESHOLD,

    "feature_columns": FEATURE_COLUMNS,

    "config": CONFIG
}


joblib.dump(
    COMPLETE_PIPELINE,
    os.path.join(
        SAVE_DIR,
        "riskgraph_pipeline.pkl"
    )
)

print("[OK] Complete pipeline saved")


# ============================================================
# 15. FINAL INVENTORY
# ============================================================

print("\n" + "=" * 70)
print("RISKGRAPH MODEL PACKAGE")
print("=" * 70)

for filename in sorted(
    os.listdir(SAVE_DIR)
):

    filepath = os.path.join(
        SAVE_DIR,
        filename
    )

    size_mb = (
        os.path.getsize(filepath)
        / (1024 * 1024)
    )

    print(
        f"{filename:<45}"
        f"{size_mb:>8.2f} MB"
    )

print("=" * 70)

print(
    f"\nSaved to:\n{SAVE_DIR}"
)

print(
    f"\nDecision threshold: "
    f"{FINAL_THRESHOLD:.4f}"
)

print(
    "\nDONE."
)

In [ ]:
import shutil

shutil.make_archive(
    "/kaggle/working/riskgraph_final_model",
    "zip",
    "/kaggle/working/riskgraph_final_model"
)

print("Model package zipped successfully!")
print("/kaggle/working/riskgraph_final_model.zip")

In [ ]:
import os
from IPython.display import FileLink, display

# 1. Force the correct working directory
os.chdir('/kaggle/working/')

# 2. Set your file name here (Do NOT include absolute paths like /kaggle/working/ here)
zip_filename = '/kaggle/working/riskgraph_final_model.zip' 

# 3. Verify and display
if os.path.exists(zip_filename):
    print(f"✅ Success! File found. Click the link below:")
    display(FileLink(zip_filename))
else:
    print(f"❌ Error: '{zip_filename}' was not found in /kaggle/working/")
    print("\nHere are the files that actually exist in your directory:")
    print(os.listdir('.'))


In [ ]:
import os
import shutil
from IPython.display import FileLink, display

src = "/kaggle/working/riskgraph_final_model"

# Make sure the model folder exists
print("Model folder exists:", os.path.exists(src))
print("Files:", os.listdir(src))

# Create ZIP
zip_base = "/kaggle/working/riskgraph_final_model"
zip_path = shutil.make_archive(
    zip_base,
    "zip",
    root_dir="/kaggle/working",
    base_dir="riskgraph_final_model"
)

print("\nZIP created:")
print(zip_path)
print(f"Size: {os.path.getsize(zip_path) / (1024**2):.2f} MB")

# Create a direct download link inside the notebook
display(FileLink(zip_path, result_html_prefix="⬇️ Download model: "))